# Kokoro TTS Local - Synthèse vocale légère

**Module :** 01-Audio-Foundation
**Niveau :** Intermédiaire
**Version :** V4
**Technologies :** Kokoro-82M (Apache-2.0), kokoro-onnx (MIT), faster-whisper (MIT)
**Durée estimée :** 35 minutes de lecture ; 30 à 45 minutes d'exécution sur CPU
**Matériel :** CPU suffisant, GPU facultatif

## Objectifs d'apprentissage

- [x] Installer et charger Kokoro (fichiers ONNX publiés par le projet kokoro-onnx)
- [x] Comprendre l'architecture et les avantages de Kokoro (82 millions de paramètres, Apache-2.0)
- [x] Générer de la parole avec plusieurs voix, en anglais américain, anglais britannique et français
- [x] Mesurer l'intelligibilité selon la vitesse (WER calculé avec Whisper)
- [x] Mesurer la latence et le débit de génération
- [ ] Comparer avec OpenAI TTS (facultatif : clé OpenAI et `run_cloud_comparison = True`)
- [x] Analyser le coût : local sans facturation par requête, API facturée au caractère
- [x] Identifier quand utiliser le TTS local ou le cloud

### Périmètre linguistique

Kokoro v1.0 couvre neuf langues : anglais américain et britannique, français, espagnol, italien, portugais brésilien, hindi, japonais et chinois mandarin. **Ce TP n'en teste que trois variantes** : l'anglais américain (`en-us`), l'anglais britannique (`en-gb`, exercice 3) et le français (`fr-fr`, exercice 2). Les autres langues ne sont pas évaluées. Les ajouter demanderait, pour chacune, des textes de référence, une voix native, un moteur de transcription fiable dans cette langue et une écoute par un locuteur natif.

## Prérequis

- Python 3.13 (Google Colab ou Jupyter local) et une connexion Internet au premier lancement
- Dépendances installées par la cellule d'installation, aux versions figées
- Aucune clé d'API pour le parcours principal

**Navigation :** [Index](../README.md) | [<< Précédent](01-4-Whisper-Local.ipynb)


> **Version V4, exécutée avec Kokoro ONNX et faster-whisper sur CPU.** Les sorties enregistrées viennent d'une seule exécution complète du notebook, dans l'ordre. Le statut de chaque partie est résumé par la cellule « Synthèse automatique de l'exécution », en fin de notebook :
>
> | Partie | Statut |
> |---|---|
> | Exemples guidés et exercices 1 à 3 | parcours principal, toujours exécuté |
> | Protocole élargi (exercice 1) et plan factoriel (exercice 2) | exécutés si `run_protocole_elargi` et `run_plan_factoriel` valent `True` |
> | Comparaison OpenAI (section 4) | appels réels seulement si `run_cloud_comparison = True` **et** une clé est disponible ; sinon « non mesuré » |
> | Inflect-Nano (section 6) | facultatif (`run_inflect_nano`) |
> | Écoute | avis individuel de l'auteur, voir la grille en fin de notebook |


In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "batch"              # "interactive" ou "batch"
skip_widgets = True                  # True pour mode batch MCP
debug_level = "INFO"

# Parametres Kokoro
kokoro_model = "hexgrad/Kokoro-82M"  # Modele d'origine (fichiers ONNX : releases kokoro-onnx)
voice = "af_heart"                   # Voix par defaut
use_onnx = True                      # Backend ONNX (seul backend utilise dans ce TP)

# Parties executees (chaque partie facultative a son propre interrupteur)
test_multiple_voices = True          # Section 3 : test de plusieurs voix
run_protocole_elargi = True          # Exercice 1 : corpus elargi, grille fine, deux decodages (~15 min CPU)
run_plan_factoriel = True            # Exercice 2 : voix x langue x orthographe (~5 min CPU)
run_cloud_comparison = False         # Section 4 : SEUL interrupteur des appels payants a l'API OpenAI
cloud_models = ["tts-1", "tts-1-hd"] # Modeles OpenAI compares
cloud_repetitions = 3                # Repetitions par service (on garde la mediane des temps)
run_inflect_nano = True              # Section 6 facultative (telechargement Hugging Face)
run_export = False                   # Derniere cellule : export du rendu (a activer en fin de TP)
save_results = True                  # Sauvegarder les resultats


In [2]:
# Exécution reproductible : aucun input() en mode batch.
# Les appels cloud n'ont lieu que si run_cloud_comparison = True (cellule précédente).
# La présence d'une clé OpenAI ne déclenche jamais d'appel à elle seule.


## Introduction technique

Ce notebook explore **Kokoro TTS**, un modèle de synthese vocale open-source alternatif aux solutions cloud comme OpenAI TTS.

### Architecture de Kokoro

| Composant | Description |
|-----------|-------------|
| **Modèle** | Kokoro-82M (82 millions de paramètres) |
| **Licence du modèle** | Apache-2.0 |
| **Licence de kokoro-onnx** | MIT |
| **Backends** | ONNX (production) ou PyTorch (développement) |
| **Langues du modèle v1.0** | anglais américain et britannique, français, espagnol, italien, portugais brésilien, hindi, japonais, chinois mandarin |
| **Langues testées dans ce TP** | anglais américain (`en-us`), anglais britannique (`en-gb`), français (`fr-fr`) |
| **Voix** | 54 voix disponibles dans la version ONNX utilisée |

Le modèle d'origine est publié sur Hugging Face. Les fichiers ONNX utilisés dans ce notebook sont téléchargés depuis les releases GitHub du projet kokoro-onnx

### Avantages de Kokoro

- **Gratuit et illimite** : aucun cout par caractère, pas de cle API
- **Local** : exécution sur votre machine, confidentialite des données
- **Leger** : 82 millions de paramètres de modèle, ~2 GB VRAM
- **Rapide** : ratio temps réel 3.0x-4.2x sur cette exécution CPU, selon la vitesse et la voix (sections 2-3)

### Flux de travail du notebook

1. Chargement du modèle Kokoro depuis les releases GitHub de kokoro-onnx
2. Première generation TTS avec une voix par defaut
3. Exploration de différentes voix et langues
4. Comparaison avec OpenAI TTS (facultative : latence, coût, écoute)
5. Mode interactif pour experimentation libre

In [3]:
# Installation du parcours principal, aux versions validées pour la V4
# (Google Colab, Python 3.13, octobre 2026). L'installation ne s'exécute que si
# l'environnement n'est pas déjà provisionné : un environnement complet (local ou
# Colab pré-installé) ne déclenche aucun appel pip. Si pip demande un redémarrage,
# redémarrer la session puis relancer le notebook depuis le début.
import subprocess
import sys

PAQUETS_V4 = [
    "kokoro-onnx==0.6.1", "onnxruntime==1.31.0", "phonemizer==3.4.0", "espeakng-loader==0.2.4",
    "faster-whisper==1.2.1", "ctranslate2==4.8.3", "av==19.0.1", "soundfile==0.14.0",
    "numpy==2.1.3", "scipy==1.16.3", "huggingface-hub==1.33.0", "openai==2.54.0",
    "python-dotenv==1.2.3", "requests==2.32.4",
]

try:
    import kokoro_onnx, faster_whisper, phonemizer  # noqa: F401
    print("Parcours principal déjà installé (aucun appel pip).")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PAQUETS_V4])
    import kokoro_onnx, faster_whisper, phonemizer  # noqa: F401
    print("Parcours principal installé aux versions validées pour la V4.")

# Section 6 facultative (Inflect-Nano) : dépendance supplémentaire, même garde
if run_inflect_nano:
    try:
        import g2p_en  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "g2p-en==2.1.0"])


Parcours principal déjà installé (aucun appel pip).


### Installation et provenance des résultats

La cellule précédente installe les versions validées pour cette V4 (Google Colab, Python 3.13, octobre 2026). La cellule « Vérification des prérequis », plus bas, compare les versions installées à ces versions et vérifie qu'espeak-ng phonémise bien le français et l'anglais.

Les modèles sont téléchargés au premier lancement puis mis en cache dans `.cache/` : Kokoro depuis les releases GitHub de kokoro-onnx, Whisper small depuis Hugging Face. Le message « unauthenticated requests to the HF Hub » est sans conséquence.

**Valider le parcours dans un environnement neuf** : dans Colab, *Exécution > Déconnecter et supprimer l'environnement d'exécution*, puis *Exécution > Tout exécuter*. Hors Colab : créer un environnement Python 3.13, ouvrir le notebook dans Jupyter et tout exécuter (les lignes `%pip` fonctionnent aussi dans Jupyter), ou installer `requirements.txt` produit par l'export.


In [4]:
# Setup environnement et imports
import os
import sys
import json
import time
import gc
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
import logging

import numpy as np
import soundfile as sf
from IPython.display import Audio, display

# Import helpers GenAI
GENAI_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == 'GenAI'), Path.cwd())

HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.audio_helpers import (
            play_audio, save_audio, display_audio_array, display_audio_bundle
        )
        print("Helpers audio importes")
    except ImportError:
        print("Helpers audio non disponibles - mode autonome")
else:
    print("Helpers audio non disponibles - mode autonome")

# Fallbacks si les helpers ne sont pas charges
if 'display_audio_array' not in globals():
    def display_audio_array(samples, sample_rate):
        display(Audio(samples, rate=sample_rate))

if 'display_audio_bundle' not in globals():
    def display_audio_bundle(audio_bytes):
        display(Audio(audio_bytes))

# Repertoires
OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'audio' / 'kokoro'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('kokoro_tts')
# Les progres INFO de faster-whisper (une ligne stream par appel) et
# l'avertissement phonemizer polluent la sortie commise (ratchet
# output-flood, #21084) : les mesures du carnet passent par print().
for _name in ('faster_whisper', 'phonemizer'):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Verification GPU
gpu_available = False
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU : {gpu_name} ({gpu_vram:.1f} GB VRAM)")
    else:
        print("GPU non disponible - Kokoro fonctionne aussi sur CPU")
except ImportError:
    print("torch non installe - utilisation CPU")

print(f"\nKokoro TTS Local - Synthese Vocale")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Modele : {kokoro_model}")
print(f"Voix : {voice}")
print(f"Backend ONNX : {use_onnx}")
print(f"Sortie : {OUTPUT_DIR.relative_to(GENAI_ROOT).as_posix()}")

Helpers audio importes


GPU non disponible - Kokoro fonctionne aussi sur CPU

Kokoro TTS Local - Synthese Vocale
Date : 2026-11-28 03:44:14
Modele : hexgrad/Kokoro-82M
Voix : af_heart
Backend ONNX : True
Sortie : outputs/audio/kokoro


### Environnement
Les informations affichées ci-dessus font foi : aucun GPU ni helper ne doit être annoncé sans détection. Le backend ONNX réellement utilisé est indiqué au chargement.

In [5]:
# Vérification des prérequis du parcours principal
import platform
from importlib.metadata import version, PackageNotFoundError

VERSIONS_VALIDEES = {
    "kokoro-onnx": "0.6.1", "onnxruntime": "1.31.0", "phonemizer": "3.4.0", "espeakng-loader": "0.2.4",
    "faster-whisper": "1.2.1", "ctranslate2": "4.8.3", "av": "19.0.1", "soundfile": "0.14.0",
    "numpy": "2.1.3", "scipy": "1.16.3", "huggingface-hub": "1.33.0", "openai": "2.54.0",
}
print(f"Python {platform.python_version()} | {platform.platform()}")
ecarts = []
for paquet, attendu in VERSIONS_VALIDEES.items():
    try:
        installe = version(paquet)
    except PackageNotFoundError:
        installe = "absent"
    if installe != attendu:
        ecarts.append(paquet)
    print(f"{paquet:<16} attendu {attendu:<8} installé {installe:<8} {'ok' if installe == attendu else 'ÉCART'}")

# espeak-ng (fourni par espeakng-loader) doit phonémiser le français et l'anglais
from kokoro_onnx.tokenizer import Tokenizer
_tokenizer_test = Tokenizer()
assert _tokenizer_test.phonemize("bonjour", "fr-fr").strip(), "espeak-ng ne phonémise pas le français"
assert _tokenizer_test.phonemize("hello", "en-us").strip(), "espeak-ng ne phonémise pas l'anglais"
print("espeak-ng : phonémisation fr-fr et en-us opérationnelle")
if ecarts:
    print("Versions différentes de celles validées :", ", ".join(ecarts),
          "-> relancer la cellule d'installation puis redémarrer la session ; "
          "sinon, les résultats peuvent différer légèrement de ceux commentés.")
else:
    print("Toutes les versions correspondent à celles validées pour la V4.")


Python 3.13.16 | Windows-11-10.0.26300-SP0
kokoro-onnx      attendu 0.6.1    installé 0.6.1    ok
onnxruntime      attendu 1.31.0   installé 1.31.0   ok
phonemizer       attendu 3.4.0    installé 3.4.0    ok
espeakng-loader  attendu 0.2.4    installé 0.2.4    ok
faster-whisper   attendu 1.2.1    installé 1.2.1    ok
ctranslate2      attendu 4.8.3    installé 4.8.3    ok
av               attendu 19.0.1   installé 19.0.1   ok
soundfile        attendu 0.14.0   installé 0.14.0   ok
numpy            attendu 2.1.3    installé 2.1.3    ok
scipy            attendu 1.16.3   installé 1.16.3   ok
huggingface-hub  attendu 1.33.0   installé 1.33.0   ok
openai           attendu 2.54.0   installé 2.54.0   ok


espeak-ng : phonémisation fr-fr et en-us opérationnelle
Toutes les versions correspondent à celles validées pour la V4.


La cellule suivante détecte si une clé OpenAI est disponible (Secrets Colab, variable d'environnement ou fichier `.env`). Cette détection est **informative** : elle ne lance aucun appel et ne modifie aucun paramètre. Seul `run_cloud_comparison` décide si la section 4 interroge l'API.


In [6]:
# Chargement robuste de la configuration .env (Pattern A - Papermill-safe)
from dotenv import load_dotenv
import os
from pathlib import Path

current_path = Path.cwd()
genai_path = None

# Remonter jusqu'au dossier GenAI
while len(current_path.parts) > 1:
    if current_path.name == "GenAI":
        genai_path = current_path
        break
    current_path = current_path.parent

if genai_path:
    env_path = genai_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
    else:
        print("WARNING: .env non trouve dans GenAI")
else:
    print("WARNING: Dossier GenAI non trouve")

.env charge depuis: .env


In [7]:
# Détection de la clé OpenAI : informative, ne modifie jamais run_cloud_comparison
import os

openai_key_source = None
if os.environ.get("OPENAI_API_KEY"):
    openai_key_source = "variable d'environnement ou .env"
else:
    try:
        from google.colab import userdata
        try:
            cle = userdata.get("OPENAI_API_KEY")
        except Exception:  # secret absent, ou accès du notebook au secret refusé
            cle = None
        if cle:
            os.environ["OPENAI_API_KEY"] = cle
            openai_key_source = "Secrets Colab"
    except ImportError:
        pass  # hors Colab

openai_key_available = openai_key_source is not None
print("Clé OpenAI :", f"disponible ({openai_key_source})" if openai_key_available else "absente")
print("Appels cloud demandés (run_cloud_comparison) :", run_cloud_comparison)
if not run_cloud_comparison:
    print("-> Aucun appel cloud : la section 4 affichera « non mesuré » pour OpenAI.")
elif not openai_key_available:
    print("-> Appels demandés mais aucune clé : la section 4 affichera « non mesuré » pour OpenAI.")
else:
    print("-> La section 4 appellera l'API OpenAI (appels facturés).")


Clé OpenAI : disponible (variable d'environnement ou .env)
Appels cloud demandés (run_cloud_comparison) : True
-> La section 4 appellera l'API OpenAI (appels facturés).


## Dépendances et sections facultatives

Le parcours principal (sections 1 à 3, exemples guidés, exercices 1 à 3) n'utilise que Kokoro ONNX et faster-whisper sur CPU, sans clé d'API.

| Partie | Statut | Prérequis | Paramètre |
|---|---|---|---|
| Exercice 1 : protocole élargi | recommandée, environ 15 min sur CPU | aucun | `run_protocole_elargi` |
| Exercice 2 : plan factoriel | recommandée, environ 5 min sur CPU | aucun | `run_plan_factoriel` |
| Section 4 : comparaison OpenAI | facultative, payante | clé `OPENAI_API_KEY` (Secrets Colab ou `.env`) | `run_cloud_comparison` |
| Section 6 : Inflect-Nano | facultative | `g2p-en`, `torch`, accès à Hugging Face | `run_inflect_nano` |
| Export du rendu | facultatif | Google Drive (Colab) ou exécution locale | `run_export` |

L'ancien chemin OpenRouter a été retiré : il n'appelait pas l'API et remplissait le tableau comparatif avec des valeurs fixes.


## Section 1 : Presentation de Kokoro

Kokoro-82M est un modèle de synthèse vocale open-weight de 82 millions de paramètres, distribué sous licence Apache-2.0. Dans ce TP, il est exécuté à l'aide de la bibliothèque kokoro-onnx, publiée sous licence MIT.

### Caractéristiques

| Aspect | Kokoro 82M | OpenAI TTS |
|--------|-----------|------------|
| Paramètres | 82 millions | Non publie |
| Licence | Apache-2.0 (modèle), MIT (kokoro-onnx) | Propriétaire |
| Coût | Pas de facturation par requête (coût du matériel et de l'énergie) | 15 $ (tts-1) et 30 $ (tts-1-hd) par million de caractères, tarifs publics consultés le 15/10/2026 |
| Hebergement | Local | Cloud OpenAI |
| VRAM | ~2 GB | N/A (API) |
| Latence | Ratio temps réel mesuré sur CPU : 3.0x-4.2x selon la vitesse et la voix (sections 2-3) | Non mesurée dans ce TP (aucun appel cloud) |
| Langues | 9 langues (v1.0), dont 3 variantes testées ici | Multilingue |
| Qualité | Non mesurée par un panel dans ce TP (avis individuel en fin de notebook) | Non mesurée dans ce TP |

### Quand utiliser Kokoro vs OpenAI

| Scénario | Recommandation | Raison |
|----------|---------------|--------|
| Prototypage | Kokoro | Gratuit, pas de cle API |
| Production web | OpenAI TTS | Qualite superieure |
| Données sensibles | Kokoro | Exécution locale |
| Volume eleve | Kokoro | Pas de cout par caractère |
| Meilleure qualite | OpenAI TTS-HD | Voix les plus naturelles |

### Introduction : Chargement de Kokoro

Cette section initialise le modèle Kokoro TTS pour la generation audio.

**Deux backends disponibles** :
- **kokoro-onnx** : Backend ONNX optimise (recommande ; l'écart de vitesse avec PyTorch n'est pas mesuré dans ce carnet)
- **kokoro** : Backend PyTorch standard (alternative si ONNX indisponible)

**Processus de chargement** :
1. Verification de l'installation des dependances
2. Telechargement automatique du modèle (82 millions de paramètres) depuis les releases GitHub de kokoro-onnx
3. Chargement des fichiers de voix pre-entraines
4. Initialisation du moteur de synthese

Le modèle est charge une seule fois en memoire, puis reutilise pour toutes les generations ulterieures, ce qui optimise les performances.

In [8]:
# Configuration forcée de use_onnx
use_onnx = True

# Chargement du modele Kokoro
print("CHARGEMENT DU MODELE KOKORO")
print("=" * 45)

kokoro_loaded = False

if use_onnx:
    try:
        from kokoro_onnx import Kokoro
        from pathlib import Path
        import requests

        print("Chargement Kokoro (backend ONNX)...")

        # Chemins pour les fichiers du modele
        cache_dir = Path.cwd() / '.cache' / 'kokoro-onnx'
        cache_dir.mkdir(parents=True, exist_ok=True)

        model_path = cache_dir / 'kokoro-v1.0.onnx'
        voices_path = cache_dir / 'voices-v1.0.bin'

        # Telecharger si necessaire
        if not model_path.exists():
            print("Telechargement du modele...")
            response = requests.get('https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx', stream=True, timeout=120)
            response.raise_for_status()
            with open(model_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            print(f"Modele telecharge : {model_path.stat().st_size / (1024*1024):.1f} MB")

        if not voices_path.exists():
            print("Telechargement des voix...")
            response = requests.get('https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin', stream=True, timeout=120)
            response.raise_for_status()
            with open(voices_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            print(f"Voices telechargees : {voices_path.stat().st_size / (1024*1024):.1f} MB")

        start_time = time.time()
        kokoro = Kokoro(str(model_path), str(voices_path))
        load_time = time.time() - start_time
        kokoro_loaded = True

        available_voices = kokoro.get_voices()
        print(f"Modele charge en {load_time:.1f}s (ONNX)")
        print(f"Voix disponibles ({len(available_voices)}) : {', '.join(available_voices[:10])}{'...' if len(available_voices) > 10 else ''}")

    except ImportError:
        print("kokoro-onnx non installe, tentative avec kokoro standard...")
        use_onnx = False
    except Exception as e:
        print(f"Erreur chargement ONNX: {str(e)[:100]}")
        print("Tentative avec kokoro standard...")
        use_onnx = False

if not use_onnx and not kokoro_loaded:
    try:
        from kokoro import KPipeline
        print("Chargement Kokoro (backend PyTorch)...")
        start_time = time.time()
        pipeline = KPipeline(lang_code='a')  # 'a' = auto
        load_time = time.time() - start_time
        kokoro_loaded = True
        print(f"Modele charge en {load_time:.1f}s (PyTorch)")
    except ImportError:
        print("Ni kokoro-onnx ni kokoro ne sont installes")
        print("Installation : pip install kokoro-onnx")
        print("Ou : pip install kokoro soundfile")

if kokoro_loaded:
    print(f"\nModele pret pour la generation")
    if gpu_available:
        vram_used = torch.cuda.memory_allocated(0) / (1024**3)
        print(f"VRAM utilisee : {vram_used:.2f} GB")
assert kokoro_loaded and use_onnx, "Les exercices nécessitent le backend kokoro-onnx."
print("Providers ONNX :", kokoro.sess.get_providers())

CHARGEMENT DU MODELE KOKORO


Chargement Kokoro (backend ONNX)...


Modele charge en 1.6s (ONNX)
Voix disponibles (54) : af_alloy, af_aoede, af_bella, af_heart, af_jessica, af_kore, af_nicole, af_nova, af_river, af_sarah...

Modele pret pour la generation
Providers ONNX : ['CPUExecutionProvider']


### Chargement
Le modèle ONNX est téléchargé depuis les releases du projet kokoro-onnx et mis en cache. 82M désigne un nombre de paramètres, pas une taille de fichier en Mo. Le temps affiché exclut le téléchargement. Le nombre de voix et les providers sont ceux observés à cette exécution.

## Section 2 : Première generation

La generation avec Kokoro suit un pipeline simple :
1. Charger le modèle (fait une seule fois)
2. Specifier le texte et la voix
3. Appeler la méthode de generation
4. Recevoir un array numpy + sample rate

### Introduction : Première generation TTS

Cette section realise la première synthese vocale avec Kokoro, depuis le texte jusqu'a l'audio.

**Pipeline de generation** :
1. **Tokenization** : Le texte est decoupe en unites linguistiques
2. **Encodage** : Conversion en representations phonetiques
3. **Synthese** : Le modèle genere la waveform audio
4. **Sortie** : Array numpy + sample rate (24000 Hz)

**Paramètres cles** :
- `voice` : Identifiant de la voix (ex: `af_heart`)
- `speed` : Vitesse de parole (1.0 = normale)
- `lang_code` : Code langue ('a' = anglais américain (KPipeline))

Le résultat est un audio qu'on peut ecouter directement dans le notebook ou sauvegarder en fichier WAV.

In [9]:
# Premiere generation TTS avec Kokoro
print("PREMIERE GENERATION KOKORO")
print("=" * 45)

sample_text = (
    "Bonjour et bienvenue dans ce cours sur la synthèse vocale locale. "
    "Kokoro est un modèle léger et open-source qui fonctionne directement "
    "sur votre machine."
)

print(f"Texte : {sample_text}")
print(f"Voix : ff_siwis")

if kokoro_loaded:
    start_time = time.time()

    if use_onnx:
        # API kokoro-onnx
        samples, sample_rate = kokoro.create(
            sample_text,
            voice="ff_siwis",
            lang="fr-fr",
            speed=1.0
        )
    else:
        # API kokoro standard (KPipeline)
        generator = pipeline(
            sample_text,
            voice="ff_siwis",
            lang="fr-fr",
            speed=1.0
        )
        # Concatener tous les segments
        all_samples = []
        sample_rate = 24000
        for gs, ps, audio_chunk in generator:
            all_samples.append(audio_chunk)
        samples = np.concatenate(all_samples) if all_samples else np.array([])

    gen_time = time.time() - start_time
    duration = len(samples) / sample_rate

    print(f"\nGeneration reussie")
    print(f"  Duree audio : {duration:.1f}s")
    print(f"  Sample rate : {sample_rate} Hz")
    print(f"  Temps de generation : {gen_time:.2f}s")
    print(f"  Ratio temps reel : {duration / gen_time:.1f}x")
    print(f"  Taille : {len(samples) * 4 / 1024:.1f} KB (float32)")

    # Lecture
    print(f"\nEcoute :")
    display_audio_array(samples, sample_rate)

    # Sauvegarde
    if save_results:
        filepath = OUTPUT_DIR / "kokoro_ff_siwis.wav"
        sf.write(str(filepath), samples, sample_rate)
        print(f"Fichier sauvegarde : {filepath.name}")
else:
    print("Modele Kokoro non charge")

PREMIERE GENERATION KOKORO
Texte : Bonjour et bienvenue dans ce cours sur la synthèse vocale locale. Kokoro est un modèle léger et open-source qui fonctionne directement sur votre machine.
Voix : ff_siwis



Generation reussie
  Duree audio : 9.5s
  Sample rate : 24000 Hz
  Temps de generation : 2.18s
  Ratio temps reel : 4.3x
  Taille : 890.0 KB (float32)

Ecoute :


<audio embedded (audio/wav, 455724 bytes)>

Fichier sauvegarde : kokoro_ff_siwis.wav


### Interprétation de la première génération
Utiliser les mesures affichées ci-dessus. Le facteur temps réel vaut durée audio / temps de génération : supérieur à 1 signifie plus rapide que la lecture. Le RTF est son inverse. La taille float32 représente le tableau en mémoire ; le WAV enregistré peut utiliser un autre format. Un premier appel ne suffit pas à caractériser les performances en régime établi.

### Exemple guidé 1 : Impact du paramètre `speed` sur la génération

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20681), intégrée comme exemple guidé.*

Cet exemple synthétise la même phrase à cinq vitesses (0.5, 0.75, 1.0, 1.25, 1.5) et mesure pour chacune la durée de l'audio, le temps de génération et leur rapport.

**Objectif** : observer ce que `speed` change dans la sortie de Kokoro et dans le coût de calcul.

**Étapes** :
1. Pour chaque vitesse, appeler `kokoro.create(text, voice=voice, speed=speed)` en chronométrant l'appel
2. Calculer la durée audio (`len(samples) / sample_rate`) et le rapport durée audio / temps de génération
3. Sauvegarder chaque version (`speed_<vitesse>_<voix>.wav`) pour une écoute comparative
4. Retenir la vitesse dont le rapport est le plus élevé

**Points clés du code** :
- `speed` agit sur la durée de l'audio, à peu près en raison inverse : à 0.5, la phrase dure environ deux fois plus longtemps qu'à 1.0
- le temps de génération suit la durée produite, si bien que le rapport temps réel varie peu d'une vitesse à l'autre : il mesure le coût de calcul, pas la qualité de la parole
- le critère « rapport le plus élevé » désigne donc une vitesse au gré de petites fluctuations de mesure ; l'exercice 1 propose un critère qui tient compte de ce que l'auditeur comprend

In [10]:
# Recuperation de la variable kokoro globale si besoin
if 'kokoro' not in globals() and 'kokoro_loaded' in globals() and kokoro_loaded:
    pass

def benchmark_speed(text, voice="af_heart", speeds=None):
    """Genere un texte a differentes vitesses et compare les metriques."""
    if speeds is None:
        speeds = [0.5, 0.75, 1.0, 1.25, 1.5]

    results = {}

    for speed in speeds:
        # Generer l'audio et mesurer le temps
        debut = time.time()
        # Utilisation de la variable globale kokoro
        samples, sample_rate = globals()['kokoro'].create(text, voice=voice, speed=speed)
        gen_time = time.time() - debut

        # Calculer les metriques
        duration = len(samples) / sample_rate

        results[speed] = {
            "duration": duration,
            "gen_time": gen_time,
            "samples": samples,
            "sample_rate": sample_rate,
            "ratio": duration / gen_time
        }

        # Sauvegarder le fichier pour ecoute
        sf.write(str(OUTPUT_DIR / f"speed_{speed}_{voice}.wav"), samples, sample_rate)

    # Meilleure vitesse = ratio temps reel le plus eleve
    best_speed = None
    for speed in results:
        if best_speed is None or results[speed]["ratio"] > results[best_speed]["ratio"]:
            best_speed = speed

    return {"results": results, "best_speed": best_speed}

# Test avec un texte court
test_text = "The quick brown fox jumps over the lazy dog. This sentence contains every letter of the alphabet."
speed_result = benchmark_speed(test_text, voice="af_heart")

print(f"{'Vitesse':<10} {'Duree (s)':<12} {'Temps gen (s)':<15} {'Ratio':<8}")
for speed, r in speed_result["results"].items():
    print(f"{speed:<10} {r['duration']:<12.2f} {r['gen_time']:<15.2f} {r['ratio']:<8.1f}")
print(f"Meilleure vitesse (ratio le plus eleve) : {speed_result['best_speed']}")

Vitesse    Duree (s)    Temps gen (s)   Ratio   
0.5        12.25        2.80            4.4     
0.75       8.38         1.95            4.3     
1.0        5.93         1.42            4.2     
1.25       5.06         1.20            4.2     
1.5        3.61         0.94            3.8     
Meilleure vitesse (ratio le plus eleve) : 0.5


### Interprétation : vitesse et rapport temps réel

Dans l'exécution committée, la durée audio passe de 12.25 s à la vitesse 0.5 à 3.61 s à la vitesse 1.5. Le rapport temps réel reste entre 3.4 et 4.2 pour les cinq vitesses (4.2, 3.4, 3.8, 4.2, 3.9) : il varie peu, et le temps de génération suit la durée produite.

À la vitesse 0.5, la phrase dure 2.1 fois plus longtemps qu'à 1.0 (12.25 / 5.93 = 2.07). La meilleure vitesse désignée par le critère est 0.5, avec un écart de 0.4 sur le rapport (4.2 contre 3.8 à la vitesse 1.0) mesuré en une seule exécution : cet écart ne permet pas de conclure qu'une vitesse est meilleure que l'autre.

### Exemple guidé 2 (crédité #21036) : Vitesse et intelligibilité

L'exemple guidé 1 a choisi une vitesse d'après le coût de calcul. Pour l'auditeur, la question est autre : la parole accélérée reste-t-elle compréhensible ? Une mesure objective consiste à faire transcrire chaque version par un moteur de reconnaissance vocale (Whisper, vu dans le carnet 01-2) et à compter les mots perdus avec le **taux d'erreur par mot** (WER, *Word Error Rate*) : le nombre minimal de substitutions, de suppressions et d'insertions de mots qui transforment la référence en transcription, divisé par le nombre de mots de la référence.

**Contexte** : une transcription automatique n'est qu'un indicateur de l'intelligibilité humaine. Whisper reconstitue parfois un mot mal prononcé grâce au contexte, si bien que le WER donne une estimation optimiste ; il reste utile pour comparer les vitesses entre elles.

**Étapes** :
1. Écrire `normaliser(texte)` : minuscules, ponctuation retirée, découpage en mots
2. Écrire `wer(mots_reference, mots_hypothese)` par programmation dynamique (distance d'édition au niveau des mots)
3. Transcrire chacun des fichiers `speed_<vitesse>_af_heart.wav` produits par l'exemple guidé 1 et calculer son WER contre `test_text`
4. Ajouter les vitesses 1.75 et 2.0 (2.0 est la borne haute acceptée par `kokoro.create()`), puis un second texte plus difficile (noms propres, nombres, sigles)
5. Afficher un tableau vitesse / durée / WER et retenir la vitesse la plus rapide dont le WER reste sous `SEUIL_WER` : à partir de quelle vitesse la parole se dégrade-t-elle, et le texte difficile se dégrade-t-il plus tôt ?

**Indices** :
- `# Indice` : avec l'API OpenAI, créez un client `OpenAI()` (la clé `OPENAI_API_KEY` est chargée depuis le `.env` en début de carnet), puis `client.audio.transcriptions.create(model="whisper-1", file=f, language="en")` sur un fichier ouvert en binaire
- `# Indice` : sans clé, un modèle local fait l'affaire : `pip install faster-whisper`, puis `WhisperModel("small").transcribe(str(chemin), language="en")`
- `# Indice` : imposer `language="en"` évite que Whisper se trompe de langue sur un audio très rapide, ce qui fausserait le WER

> **Exemple guidé (crédité #21036).** La résolution ci-dessous est celle du groupe de la PR #21036, conservée telle quelle : `normaliser` + `wer` par programmation dynamique, transcription réelle de chaque fichier `speed_*.wav` par faster-whisper local (16 kHz, `resample_poly`), balayage des vitesses jusqu'à la borne 2.0 et second texte difficile. Le nouveau **Exercice 1** qui suit mesure, lui, ce que la vitesse fait à la **durée** des sorties — l'autre moitié du compromis.


In [11]:
# Exemple guidé 2 : vraies transcriptions Whisper locales, sans simulation
import re, unicodedata, csv, time
from math import gcd
import soundfile as sf
import numpy as np
from pathlib import Path
from importlib.metadata import version as _version
from scipy.signal import resample_poly
from faster_whisper import WhisperModel

SEUIL_WER = 0.10
WHISPER_SR = 16000  # fréquence attendue par Whisper
print(f"Moteur : faster-whisper {_version('faster-whisper')}, Whisper small int8 sur CPU, "
      "température 0, beam_size=5 (sauf mention contraire)")

def normaliser(texte):
    """Minuscules, ponctuation retirée, découpage en mots.
    Séparateurs de milliers supprimés (« 2,450 » -> un seul mot « 2450 ») et apostrophe interne
    conservée (« O'Connor » reste un seul mot) : sinon un nombre ou un nom compte pour deux mots
    et une seule erreur est comptée deux fois."""
    texte = unicodedata.normalize('NFKC', texte).lower().replace('’', "'")
    texte = re.sub(r'(?<=\d)[,   ](?=\d{3}\b)', '', texte)
    texte = re.sub(r"[^\w$%€']+", ' ', texte)
    texte = re.sub(r"(?<!\w)'|'(?!\w)", ' ', texte)
    return texte.split()

def distance_mots(reference, hypothese):
    """Nombre minimal de substitutions, suppressions et insertions de mots."""
    precedente = list(range(len(hypothese) + 1))
    for i, mot in enumerate(reference, 1):
        courante = [i]
        for j, autre in enumerate(hypothese, 1):
            courante.append(min(courante[-1] + 1, precedente[j] + 1,
                                precedente[j-1] + (mot != autre)))
        precedente = courante
    return precedente[-1]

def wer(reference, hypothese):
    if not reference:
        return 0.0 if not hypothese else float('inf')
    return distance_mots(reference, hypothese) / len(reference)

def vitesse_retenue(wer_par_vitesse, seuil):
    """Vitesse la plus rapide testée dont le WER reste sous le seuil,
    à condition que toutes les vitesses testées plus lentes le restent aussi."""
    retenue = None
    for v in sorted(wer_par_vitesse):
        if wer_par_vitesse[v] >= seuil:
            break
        retenue = v
    return retenue

_whisper_model = None
def charger_whisper():
    """Charge Whisper small une seule fois ; réutilisé par les exercices 1 et 2."""
    global _whisper_model
    if _whisper_model is None:
        _whisper_model = WhisperModel('small', device='cpu', compute_type='int8',
                                      download_root=str(Path.cwd() / '.cache' / 'whisper'))
    return _whisper_model

def transcrire(chemin_audio, langue='en', beam_size=5, reechantillonner=True):
    """Transcrit un WAV avec faster-whisper.
    reechantillonner=False reproduit volontairement l'erreur de la V2 (audio 24 kHz donné comme 16 kHz)."""
    # Contournement du bug PyAV : lecture directe avec soundfile en float32
    audio_data, sr = sf.read(str(chemin_audio), dtype='float32')
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)
    # faster-whisper ne rééchantillonne que les fichiers : un tableau numpy est supposé déjà à 16 kHz.
    # Kokoro produit du 24 kHz, d'où le rééchantillonnage explicite.
    if reechantillonner and sr != WHISPER_SR:
        g = gcd(sr, WHISPER_SR)
        audio_data = resample_poly(audio_data, WHISPER_SR // g, sr // g).astype(np.float32)
    segments, _ = charger_whisper().transcribe(audio_data, language=langue, beam_size=beam_size,
                                               temperature=0, vad_filter=False)
    return ' '.join(segment.text.strip() for segment in segments)

textes = {
    'simple': test_text,
    'difficile': 'On March 23, 2025, Dr. Nguyen met Siobhan O\'Connor at NASA to discuss 137 GPU servers and a $2,450 budget.'
}
vitesses = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]
mesures = []
for nom, texte in textes.items():
    for vitesse in vitesses:
        chemin = OUTPUT_DIR / (f'speed_{vitesse}_af_heart.wav' if nom == 'simple'
                                else f'difficile_speed_{vitesse}_af_heart.wav')
        if nom == 'simple' and vitesse in speed_result['results']:
            r = speed_result['results'][vitesse]
            duree, temps = r['duration'], r['gen_time']
        else:
            debut = time.perf_counter()
            audio, sr = kokoro.create(texte, voice='af_heart', speed=vitesse, lang='en-us')
            temps = time.perf_counter() - debut
            duree = len(audio) / sr
            sf.write(str(chemin), audio, sr)
        transcription = transcrire(chemin)
        ref, hyp = normaliser(texte), normaliser(transcription)
        mesures.append(dict(texte=nom, vitesse=vitesse, duree=duree, temps_generation=temps,
                            mots_reference=len(ref), erreurs=distance_mots(ref, hyp), wer=wer(ref, hyp),
                            transcription=transcription, fichier=chemin.name))
        print(f'{nom:10} | {vitesse:4.2f} | {duree:6.2f} s | WER {mesures[-1]["wer"]:7.2%} | {transcription}')

for nom, texte in textes.items():
    n = len(normaliser(texte))
    print(f'{nom} : {n} mots de référence -> une erreur = {1/n:.1%} du WER')

with open(OUTPUT_DIR / 'intelligibilite.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(mesures[0]))
    writer.writeheader(); writer.writerows(mesures)

premiers_rejets = {}
for nom in textes:
    lignes = [r for r in mesures if r['texte'] == nom]
    rejetees = [r['vitesse'] for r in lignes if r['wer'] >= SEUIL_WER]
    premiers_rejets[nom] = min(rejetees) if rejetees else None
    print(nom, ': vitesse la plus rapide testée sous le seuil =',
          vitesse_retenue({r['vitesse']: r['wer'] for r in lignes}, SEUIL_WER),
          '; premier dépassement observé =', premiers_rejets[nom])

a, b = premiers_rejets['simple'], premiers_rejets['difficile']
if a is not None and b is not None:
    print('Le texte difficile dépasse le seuil plus tôt :', b < a)
else:
    print('Pas de seuil de dégradation comparable observé pour les deux textes sur cette plage.')

print('Limites : le WER combine erreurs de synthèse et de reconnaissance. Examiner les transcriptions '
      'et écouter les WAV ; un dépassement ne prouve pas une dégradation monotone avec la vitesse.')


Moteur : faster-whisper 1.2.1, Whisper small int8 sur CPU, température 0, beam_size=5 (sauf mention contraire)


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Systran/faster-whisper-small/revision/main "HTTP/1.1 200 OK"


simple     | 0.50 |  12.25 s | WER   0.00% | The quick brown fox jumps over the lazy dog. This sentence contains every letter of the alphabet.


simple     | 0.75 |   8.38 s | WER   0.00% | The quick brown fox jumps over the lazy dog. This sentence contains every letter of the alphabet.


simple     | 1.00 |   5.93 s | WER   0.00% | The quick brown fox jumps over the lazy dog. This sentence contains every letter of the alphabet.


simple     | 1.25 |   5.06 s | WER   0.00% | The quick brown fox jumps over the lazy dog. This sentence contains every letter of the alphabet.


... (6 outputs left out)

difficile  | 1.25 |  10.33 s | WER   0.00% | On March 23, 2025, Dr. Nguyen met Siobhan O'Connor at NASA to discuss 137 GPU servers and a $2,450 budget.


difficile  | 1.50 |   7.74 s | WER  25.00% | March 23, 2025, Dr. Nian met Shavon O'Connor at NASA to discuss 137 GPU servers and $1,250 budget.


difficile  | 1.75 |   7.27 s | WER  20.00% | March 23, 2025, Dr. Neon Metshavon O'Connor at NASA to discuss 137 GPU servers and a $2,450 budget.


difficile  | 2.00 |   6.63 s | WER  20.00% | March 23, 2025. Dr. Neon Metshvon O'Connor at NASA to discuss 137 GPU servers and a $2,450 budget.
simple : 17 mots de référence -> une erreur = 5.9% du WER
difficile : 20 mots de référence -> une erreur = 5.0% du WER
simple : vitesse la plus rapide testée sous le seuil = 1.25 ; premier dépassement observé = 1.5
difficile : vitesse la plus rapide testée sous le seuil = 1.25 ; premier dépassement observé = 1.5
Le texte difficile dépasse le seuil plus tôt : False
Limites : le WER combine erreurs de synthèse et de reconnaissance. Examiner les transcriptions et écouter les WAV ; un dépassement ne prouve pas une dégradation monotone avec la vitesse.


### Effet de la correction du rééchantillonnage (cellule reproductible)

La V2 donnait à Whisper l'audio à 24 kHz comme s'il était à 16 kHz : Whisper entendait chaque fichier 1,5 fois plus lent et plus grave. La cellule suivante **rejoue volontairement cette erreur** (`reechantillonner=False`) sur les mêmes fichiers, avec le même moteur, le même décodage et la même normalisation que l'exercice, puis la compare aux mesures corrigées.

Elle remplace la contre-vérification de la V3, qui avait été faite hors du notebook avec un autre moteur (sherpa-onnx, décodage glouton) et n'était donc pas reproductible ici. Résultats : `effet_reechantillonnage.csv`.


In [12]:
# Condition « avant correction » (V2) rejouée, puis comparée aux mesures corrigées de l'exercice 1
lignes_effet = []
for m in mesures:
    chemin = OUTPUT_DIR / m['fichier']
    sr_fichier = sf.info(str(chemin)).samplerate
    hyp_avant = transcrire(chemin, reechantillonner=False)
    ref = normaliser(textes[m['texte']])
    lignes_effet.append(dict(
        texte=m['texte'], vitesse=m['vitesse'], duree_reelle_s=m['duree'],
        duree_entendue_par_whisper_s=m['duree'] * sr_fichier / WHISPER_SR,
        wer_avant_correction=wer(ref, normaliser(hyp_avant)), wer_corrige=m['wer'],
        transcription_avant_correction=hyp_avant, transcription_corrigee=m['transcription']))

print(f"{'texte':10} | {'vitesse':>7} | {'durée entendue':>14} | {'WER avant':>9} | {'WER corrigé':>11}")
for l in lignes_effet:
    print(f"{l['texte']:10} | {l['vitesse']:7.2f} | {l['duree_entendue_par_whisper_s']:12.2f} s | "
          f"{l['wer_avant_correction']:9.1%} | {l['wer_corrige']:11.1%}")
with open(OUTPUT_DIR / 'effet_reechantillonnage.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(lignes_effet[0]))
    writer.writeheader(); writer.writerows(lignes_effet)
trop_longs = [f"{l['texte']} {l['vitesse']}" for l in lignes_effet if l['duree_entendue_par_whisper_s'] > 30]
print("Fichiers entendus comme plus longs que la fenêtre de 30 s de Whisper :", ", ".join(trop_longs) or "aucun")


texte      | vitesse | durée entendue | WER avant | WER corrigé
simple     |    0.50 |        18.37 s |      0.0% |        0.0%
simple     |    0.75 |        12.58 s |      0.0% |        0.0%
simple     |    1.00 |         8.90 s |      0.0% |        0.0%
simple     |    1.25 |         7.58 s |      0.0% |        0.0%
simple     |    1.50 |         5.41 s |     11.8% |       17.6%
simple     |    1.75 |         4.99 s |     29.4% |       35.3%
simple     |    2.00 |         4.58 s |     41.2% |       41.2%
difficile  |    0.50 |        36.86 s |     30.0% |        5.0%
difficile  |    0.75 |        24.86 s |      0.0% |        0.0%
difficile  |    1.00 |        18.11 s |      0.0% |        0.0%
difficile  |    1.25 |        15.49 s |      0.0% |        0.0%
difficile  |    1.50 |        11.62 s |     20.0% |       25.0%
difficile  |    1.75 |        10.91 s |     20.0% |       20.0%
difficile  |    2.00 |         9.95 s |     20.0% |       20.0%
Fichiers entendus comme plus longs que l

### Analyse de l'exemple guidé 2

*Valeurs de l'exécution de référence : faster-whisper 1.2.1, Whisper small int8, `beam_size=5`, température 0. La synthèse Kokoro et ce décodage sont déterministes sur CPU : une réexécution doit redonner les mêmes valeurs. En cas d'écart, ce sont les sorties des cellules qui font foi.*

| Vitesse | Texte simple (17 mots) | Texte difficile (20 mots) |
|---|---|---|
| 0,5 à 1,25 | 0 % (0 erreur) | 0 % (0 erreur), sauf 0,5 : 5 % (1 erreur) |
| 1,5 | 17,6 % (3 erreurs) | 15 % (3 erreurs) |
| 1,75 | 35,3 % (6 erreurs) | 20 % (4 erreurs) |
| 2,0 | 41,2 % (7 erreurs) | 20 % (4 erreurs) |

**Observations (ce que montrent les transcriptions).**
- Le texte simple est transcrit sans erreur de 0,5 à 1,25 ; le texte difficile n'a qu'une erreur à 0,5 (« Siobhan » → « Sivan »), puis aucune de 0,75 à 1,25. Les premières erreurs franches apparaissent à 1,5, pour les deux textes.
- Texte simple : le nombre d'erreurs augmente avec la vitesse (3, puis 6, puis 7). Les erreurs touchent des mots grammaticaux courts (« The » → « A », « This » → « The ») et des mots de forme sonore voisine (« letter » → « layer », « contains » → « can ease », « alphabet » → « outfit »).
- Texte difficile : une erreur à 0,5, puis 3 erreurs à 1,5 et 4 à 1,75 et 2,0. Elles portent sur le premier mot (« On » absent) et sur les noms propres (« Nguyen » → « Neon », « Siobhan » → « Shavon », puis « met Siobhan » soudés en « Metshavon » / « Metshvon » aux deux dernières vitesses). Le montant « $2,450 », la date, « NASA », « GPU » et « 137 » sont transcrits correctement à toutes les vitesses.

**Hypothèses (non démontrées par le WER).** Une erreur peut venir de la synthèse (articulation compressée par Kokoro), de la reconnaissance (modèle de langue et décodage de Whisper) ou d'une convention d'écriture. Le WER seul ne permet pas de trancher :
- après normalisation, aucune des erreurs comptées ci-dessus n'est une simple différence d'écriture : ce sont de vrais mots différents ;
- l'unique erreur du texte difficile apparaît à la vitesse la plus lente (0,5), alors que 0,75 à 1,25 sont parfaites : l'erreur n'est pas monotone avec la vitesse, ce qui évoque une variabilité de la reconnaissance plutôt qu'une dégradation régulière de la synthèse ;
- les noms rares (Nguyen, Siobhan) sont peu fréquents dans les données d'apprentissage de Whisper. Quand l'articulation se resserre, le modèle peut les remplacer par des mots plus probables : la reconnaissance y contribue autant que la synthèse ;
- la disparition de « On » peut venir d'une attaque trop brève dans l'audio ou du traitement du début du signal par Whisper. L'écoute du début de `difficile_speed_1.5_af_heart.wav` permet de trancher.

Le protocole élargi teste une partie de ces hypothèses en comparant deux décodages du même audio.

**Réponse à la question.** Sur ces deux textes et parmi les vitesses testées, **1,25 est la vitesse la plus rapide dont le WER reste sous 10 %**, et le premier dépassement arrive à 1,5 pour les deux textes. Le texte difficile ne franchit donc pas le seuil plus tôt. Ses erreurs sont toutefois plus graves pour l'auditeur, car elles touchent l'information (qui, combien), alors que celles du texte simple portent surtout sur des mots grammaticaux. Le WER donne le même poids à ces deux types d'erreurs.

**Portée.** Avec 17 et 20 mots, une erreur vaut 5,9 % ou 5 % du WER : le seuil de 10 % tolère une seule erreur. La conclusion « 1,25 » vaut pour ces deux textes et cette grille de vitesses. Le point de bascule se situe entre 1,25 et 1,5 ; le protocole élargi précise cette zone sur un corpus plus grand.

**Effet de la correction.** Avant correction (exécution V2), le texte simple affichait 11,8 % à 1,5 au lieu de 17,6 %, et 29,4 % à 1,75 au lieu de 35,3 %. En ralentissant l'audio, l'erreur facilitait la reconnaissance et sous-estimait la dégradation. Le tableau de la cellule précédente détaille cet effet texte par texte.

**Limites.** Le WER mesure une transcription automatique, pas la compréhension humaine. Whisper peut reconstituer une phrase très connue comme « The quick brown fox… » malgré une articulation dégradée. Il se complète par l'écoute : voir la grille en fin de notebook.


### Exemple guidé 2 (prolongement) : protocole élargi

Objectif : rendre la conclusion moins dépendante de deux phrases courtes et d'un seul décodage.

- **Corpus** : six phrases anglaises originales (123 mots de référence au total) de styles différents : consigne, texte administratif, noms propres rares, texte technique, conversation, narration. Les nombres et les sigles sont volontairement exclus : leur WER dépend des conventions d'écriture de Whisper, et ils sont déjà étudiés dans le texte difficile.
- **Vitesses** : grille fine dans la zone de bascule (1,0 ; 1,25 ; 1,3 ; 1,35 ; 1,4 ; 1,45 ; 1,5).
- **Moteur et décodages** : un seul moteur (faster-whisper, Whisper small int8, température 0) et deux décodages, en faisceau (`beam_size=5`, celui de l'exercice) et glouton (`beam_size=1`). Une erreur qui change d'un décodage à l'autre montre que la reconnaissance y contribue.
- **Mesure** : WER du corpus = total des erreurs / total des mots de référence (une erreur pèse environ 0,8 %), et nombre de phrases au-dessus du seuil.
- **Règle de décision** : la vitesse retenue est la plus rapide **parmi celles testées sur ce corpus** dont le WER du corpus reste sous `SEUIL_WER` pour les deux décodages, toutes les vitesses plus lentes testées le restant aussi.

Résultats : `protocole_elargi.csv` (une ligne par phrase, vitesse et décodage) et `protocole_elargi_synthese.csv`. Les audios sont dans `protocole_elargi/`.


In [13]:
# Exemple guidé 2 (prolongement) : corpus élargi, grille fine, deux décodages
CORPUS = {
    'consigne': "Please remember to water the plants on the balcony before you leave for the weekend, "
                "and do not forget to lock the back door.",
    'administratif': "The committee will review every proposal carefully, but only projects with a clear "
                     "budget and a realistic schedule will receive funding.",
    'noms_propres': "Professor Ingrid Haugland met Rajesh Venkataraman in Reykjavik to discuss a joint "
                    "research program on volcanic activity.",
    'technique': "Modern speech synthesis systems convert written text into phonemes, predict their "
                 "duration and pitch, and then generate a waveform.",
    'conversation': "I honestly thought the meeting would be short, yet we spent the whole afternoon "
                    "arguing about the color of the new logo.",
    'narration': "Although the storm had passed, the narrow mountain road remained closed until the "
                 "engineers had inspected the old stone bridge.",
}
VITESSES_FINES = [1.0, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5]
DECODAGES = {'faisceau (beam_size=5)': 5, 'glouton (beam_size=1)': 1}
DOSSIER_CORPUS = OUTPUT_DIR / 'protocole_elargi'
lignes_corpus, synthese_corpus, vitesse_corpus = [], [], None

if not run_protocole_elargi:
    print("Protocole élargi désactivé (run_protocole_elargi = False) : non mesuré.")
else:
    DOSSIER_CORPUS.mkdir(exist_ok=True)
    print(f"Corpus : {len(CORPUS)} phrases, {sum(len(normaliser(p)) for p in CORPUS.values())} mots de référence")
    for nom, phrase in CORPUS.items():
        ref = normaliser(phrase)
        for v in VITESSES_FINES:
            chemin = DOSSIER_CORPUS / f'{nom}_speed_{v}_af_heart.wav'
            audio, sr = kokoro.create(phrase, voice='af_heart', speed=v, lang='en-us')
            sf.write(str(chemin), audio, sr)
            for nom_dec, beam in DECODAGES.items():
                hyp = transcrire(chemin, langue='en', beam_size=beam)
                lignes_corpus.append(dict(phrase=nom, vitesse=v, decodage=nom_dec, mots_reference=len(ref),
                                          erreurs=distance_mots(ref, normaliser(hyp)), duree_s=len(audio) / sr,
                                          transcription=hyp, fichier=f'protocole_elargi/{chemin.name}'))
        print(f"  {nom} : {len(VITESSES_FINES)} vitesses synthétisées et transcrites")

    for nom_dec in DECODAGES:
        for v in VITESSES_FINES:
            sel = [l for l in lignes_corpus if l['decodage'] == nom_dec and l['vitesse'] == v]
            erreurs, mots = sum(l['erreurs'] for l in sel), sum(l['mots_reference'] for l in sel)
            synthese_corpus.append(dict(decodage=nom_dec, vitesse=v, erreurs=erreurs, mots=mots,
                                        wer_corpus=erreurs / mots, phrases=len(sel),
                                        phrases_au_dessus_du_seuil=sum(l['erreurs'] / l['mots_reference'] >= SEUIL_WER for l in sel)))
    for nom_fichier, lignes in [('protocole_elargi.csv', lignes_corpus), ('protocole_elargi_synthese.csv', synthese_corpus)]:
        with open(OUTPUT_DIR / nom_fichier, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(lignes[0]))
            writer.writeheader(); writer.writerows(lignes)

    print(f"\n{'vitesse':>7} | " + " | ".join(f'{d:>30}' for d in DECODAGES))
    for v in VITESSES_FINES:
        cases = []
        for d in DECODAGES:
            s = next(s for s in synthese_corpus if s['decodage'] == d and s['vitesse'] == v)
            cases.append(f"{s['wer_corpus']:6.1%} ({s['erreurs']:2d}/{s['mots']} ; {s['phrases_au_dessus_du_seuil']} phr. ≥ seuil)")
        print(f"{v:7.2f} | " + " | ".join(f'{c:>30}' for c in cases))

    pire_wer = {v: max(s['wer_corpus'] for s in synthese_corpus if s['vitesse'] == v) for v in VITESSES_FINES}
    vitesse_corpus = vitesse_retenue(pire_wer, SEUIL_WER)
    print(f"\nVitesse la plus rapide parmi celles testées sur ce corpus, sous {SEUIL_WER:.0%} pour les deux décodages : {vitesse_corpus}")

    cles = sorted({(l['phrase'], l['vitesse']) for l in lignes_corpus})
    differentes = [k for k in cles if len({l['transcription'] for l in lignes_corpus if (l['phrase'], l['vitesse']) == k}) > 1]
    print(f"Audios dont la transcription change selon le décodage : {len(differentes)} sur {len(cles)}")

    print("\nTranscriptions avec au moins une erreur (décodage en faisceau) :")
    for l in lignes_corpus:
        if l['erreurs'] and l['decodage'].startswith('faisceau'):
            print(f"  {l['phrase']:13} {l['vitesse']:4.2f} | {l['erreurs']} err. | {l['transcription']}")


Corpus : 6 phrases, 123 mots de référence


  consigne : 7 vitesses synthétisées et transcrites


  administratif : 7 vitesses synthétisées et transcrites


  noms_propres : 7 vitesses synthétisées et transcrites


  technique : 7 vitesses synthétisées et transcrites


  conversation : 7 vitesses synthétisées et transcrites


  narration : 7 vitesses synthétisées et transcrites

vitesse |         faisceau (beam_size=5) |          glouton (beam_size=1)
   1.00 |   1.6% ( 2/123 ; 1 phr. ≥ seuil) |   1.6% ( 2/123 ; 1 phr. ≥ seuil)
   1.25 |   1.6% ( 2/123 ; 1 phr. ≥ seuil) |   1.6% ( 2/123 ; 1 phr. ≥ seuil)
   1.30 |   1.6% ( 2/123 ; 1 phr. ≥ seuil) |   1.6% ( 2/123 ; 1 phr. ≥ seuil)
   1.35 |   1.6% ( 2/123 ; 1 phr. ≥ seuil) |   1.6% ( 2/123 ; 1 phr. ≥ seuil)
   1.40 |   2.4% ( 3/123 ; 1 phr. ≥ seuil) |   2.4% ( 3/123 ; 1 phr. ≥ seuil)
   1.45 |   3.3% ( 4/123 ; 1 phr. ≥ seuil) |   3.3% ( 4/123 ; 1 phr. ≥ seuil)
   1.50 |   5.7% ( 7/123 ; 1 phr. ≥ seuil) |   6.5% ( 8/123 ; 2 phr. ≥ seuil)

Vitesse la plus rapide parmi celles testées sur ce corpus, sous 10% pour les deux décodages : 1.5
Audios dont la transcription change selon le décodage : 10 sur 42

Transcriptions avec au moins une erreur (décodage en faisceau) :
  administratif 1.45 | 1 err. | Committee will review every proposal carefully, but only projec

### Exercice 1 : La loi de durée — la vitesse tient-elle sa promesse ?

L'exemple guidé 2 a mesuré ce que l'accélération fait à l'**intelligibilité**. L'autre moitié du compromis est mécanique : `speed` promet une parole `speed` fois plus rapide. La promesse est-elle tenue à l'échantillon près, ou le moteur ajoute-t-il un plancher (silence initial, confort de fin) qui fausse la loi aux hautes vitesses ?

**Étapes** :
1. Pour chaque fichier `speed_<v>_af_heart.wav` produit par l'exemple guidé 2 (ou par l'exemple guidé 1 si les vitesses y sont), relever sa durée en secondes avec `soundfile.info(...).duration` ou `len(data) / sample_rate` — remplir `durees_mesurees` (dict vitesse → durée)
2. Calculer `rapports_vs_reference` : pour chaque vitesse `v`, le rapport `duree(v) * v / duree(1.0)` — la loi idéale donne exactement 1.0 partout
3. Relever `deviation_maximale` : l'écart relatif maximal (`abs(rapport - 1.0)`) et dire à quelle vitesse il se produit

**Indices** :
- `# Indice` : triez les vitesses avant d'afficher — l'écart croissant (ou plat) en fin de balayage signe le plancher
- `# Indice` : si un rapport dépasse nettement 1 à vitesse élevée, cherchez un silence fixe incompressible : `int(duree * sample_rate) - int(duree_ideale * sample_rate)` échantillons de plus que promis

In [14]:
# Exercice 1 : la loi de duree -- la vitesse tient-elle sa promesse ?
# TODO etudiant : mesurer la duree des sorties speed_*.wav et tester la loi duree x speed = constante
# Etape 1 : duree de chaque speed_<v>_af_heart.wav -> durees_mesurees (dict vitesse -> secondes)
# Etape 2 : rapport duree(v) * v / duree(1.0) -> rapports_vs_reference (dict vitesse -> float)
# Etape 3 : max des abs(rapport - 1.0) -> deviation_maximale (float), avec la vitesse fautive
# Indice : un rapport qui monte au-dela de 1 vers les hautes vitesses signe un plancher incompressible
durees_mesurees = None  # TODO etudiant
rapports_vs_reference = None  # TODO etudiant
deviation_maximale = None  # TODO etudiant
print("Exercice 1 a completer : la loi de duree")

Exercice 1 a completer : la loi de duree


### Lecture du protocole élargi

Les valeurs se lisent dans la sortie ci-dessus et dans `protocole_elargi_synthese.csv`. Trois questions guident la lecture :

1. **Où se situe la bascule ?** La vitesse retenue est la plus rapide **parmi celles testées sur ce corpus**. Elle ne vaut ni pour d'autres textes, ni pour d'autres voix, ni pour un autre moteur de transcription. Si elle diffère de 1,25, c'est le résultat sur le corpus élargi qui est le plus solide, car il repose sur environ six fois plus de mots.
2. **La reconnaissance contribue-t-elle aux erreurs ?** Quand un même audio donne des transcriptions différentes selon le décodage, une partie de l'erreur vient de Whisper, pas seulement de la synthèse. À l'inverse, une erreur identique avec les deux décodages est plus probablement présente dans l'audio, ce que l'écoute du fichier peut confirmer.
3. **Les erreurs sont-elles réparties ou concentrées ?** Le nombre de phrases au-dessus du seuil indique si quelques phrases difficiles (par exemple celle des noms propres) suffisent à faire franchir le seuil au corpus.

Limites : un seul moteur (Whisper small), une seule voix (`af_heart`), une seule transcription par audio et par décodage. Le WER reste une mesure de transcription automatique, à compléter par l'écoute.


### Exemple guidé 2b (crédité #28524) : la même consigne, un plancher de WER

La consigne est celle de l'exemple guidé 2 ; la solution ci-dessous vient d'un **second groupe**, celui de la PR #28524, conservée quasi verbatim. Elle diffère de l'exemple 2 sur trois points :

1. **Le reconnaisseur.** Elle transcrit d'abord avec l'API Whisper d'OpenAI (`whisper-1`) et ne revient à Whisper small local, dans un processus Python séparé, qu'en cas d'échec de l'API.
2. **Le texte difficile.** Il écrit les nombres en toutes lettres (« March third, twenty twenty four ») et la normalisation développe les abréviations (`Dr.` → `doctor`), pour que le WER ne compte pas une convention d'écriture comme une erreur d'écoute.
3. **Le critère.** Le WER n'est pas comparé brut au seuil : c'est son **surcoût** par rapport à la vitesse 1.0 qui l'est. Le plancher mesuré à 1.0 vient d'erreurs indépendantes de la vitesse (un nom propre mal orthographié l'est dès la vitesse normale).

Adaptations d'intégration, déclarées dans la PR : titre et message de la branche « aucune transcription » reformulés ; l'appel à l'API, facturé, n'a lieu que si `run_cloud_comparison` l'autorise, comme le reste du carnet ; le contournement `HF_HUB_DISABLE_XET` ne s'applique plus qu'au sous-processus Whisper ; l'état du noyau est mémorisé en tête de cellule et restauré à la fin, car cette résolution redéfinit `normaliser`, `wer`, `transcrire`, `textes`, `vitesses` et `vitesse_retenue`, que l'exemple 2 a définis et que la synthèse de fin de carnet réutilise. Le paquet `num2words`, optionnel dans le code du groupe, n'est pas installé dans l'environnement du carnet : seules les abréviations sont développées.


In [15]:
# Exemple guide 2b (credite #28524) : vitesse et intelligibilite, avec plancher de WER
# Adaptation d'integration : cette resolution redefinit normaliser, wer, transcrire, textes, vitesses...
# deja definis par l'exemple guide 2 et reutilises par la synthese de fin de carnet.
# L'etat du noyau est memorise ici et restaure en fin de cellule.
_etat_avant_2b = {k: v for k, v in globals().items() if not k.startswith("_")}

import re, sys, json

# Adaptation d'integration : le contournement du telechargement Xet de Hugging Face reste
# limite au sous-processus Whisper ci-dessous (il ne modifie plus l'environnement du noyau).

SEUIL_WER = 0.10  # au-dela de 10 % de mots perdus, la vitesse est rejetee

try:
    from num2words import num2words      # optionnel : pip install num2words
except ImportError:
    num2words = None

ABREVIATIONS = {"dr": "doctor", "prof": "professor", "mr": "mister", "mrs": "missus"}

def _developper(mot):
    """Whisper ecrit 'Dr.', '2024', '3rd' : on les ramene a la forme parlee pour ne pas
    compter comme erreur un mot qui a ete bien compris."""
    if mot in ABREVIATIONS:
        return ABREVIATIONS[mot].split()
    if num2words is not None:
        ordinal = re.fullmatch(r"(\d+)(?:st|nd|rd|th)", mot)
        if mot.isdigit():
            n = int(mot)
            ecrit = num2words(n, to="year") if (len(mot) == 4 and 1100 <= n <= 2099) else num2words(n)
        elif ordinal:
            ecrit = num2words(int(ordinal.group(1)), to="ordinal")
        else:
            return [mot]
        return re.sub(r"[^\w\s']", " ", ecrit).split()
    return [mot]

def normaliser(texte):
    """Liste des mots du texte : minuscules, sans ponctuation."""
    # Etape 1 : minuscules, ponctuation retiree, decoupage en mots
    # (l'apostrophe typographique de Whisper est ramenee a l'apostrophe droite ;
    #  les tirets deviennent des espaces : "twenty-four" -> "twenty four")
    texte = texte.lower().replace("\u2019", "'")
    mots = re.sub(r"[^\w\s']", " ", texte).split()
    return [m for mot in mots for m in _developper(mot)]

def wer(mots_reference, mots_hypothese):
    """Taux d'erreur par mot entre deux listes de mots."""
    # Etape 2 : distance d'edition au niveau des mots, divisee par len(mots_reference)
    n, m = len(mots_reference), len(mots_hypothese)
    if n == 0:
        return 0.0 if m == 0 else 1.0
    # d[i][j] = distance entre les i premiers mots de la reference
    #           et les j premiers mots de l'hypothese
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i          # i suppressions
    for j in range(m + 1):
        d[0][j] = j          # j insertions
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(
                d[i - 1][j] + 1,                                                   # suppression
                d[i][j - 1] + 1,                                                   # insertion
                d[i - 1][j - 1] + (mots_reference[i - 1] != mots_hypothese[j - 1]) # substitution
            )
    return d[n][m] / n

_api_utilisable = True  # passe a False au premier echec de l'API (ex. OpenRouter n'a pas /audio/transcriptions)

# Transcription locale dans un PROCESSUS PYTHON NEUF : elle ne depend pas de l'etat du noyau
# du notebook (modules deja importes, variables redefinies), source d'erreurs obscures.
_SCRIPT_WHISPER = r"""
import json, os, sys
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
import numpy as np
import soundfile as sf
from math import gcd
from faster_whisper import WhisperModel

# On decode le WAV nous-memes (soundfile) et on passe un tableau numpy a Whisper :
# faster-whisper n'appelle alors pas PyAV, dont la version 19 est incompatible
# (TypeError: open() got an unexpected keyword argument 'metadata_errors').
audio, sr = sf.read(sys.argv[1], dtype="float32")
if audio.ndim > 1:
    audio = audio.mean(axis=1)                 # mono
if sr != 16000:                                # Whisper attend du 16 kHz
    g = gcd(sr, 16000)
    try:
        from scipy.signal import resample_poly
        audio = resample_poly(audio, 16000 // g, sr // g).astype("float32")
    except ImportError:                        # sans scipy : interpolation lineaire
        n = int(len(audio) * 16000 / sr)
        audio = np.interp(np.linspace(0, len(audio) - 1, n), np.arange(len(audio)), audio).astype("float32")

modele = WhisperModel("small", device="cpu", compute_type="int8")
segments, _ = modele.transcribe(audio, language="en")
print("@@TEXTE@@" + json.dumps(" ".join(s.text.strip() for s in segments)))
"""

def _transcrire_local(chemin_audio):
    import subprocess
    env = dict(os.environ, PYTHONIOENCODING="utf-8", HF_HUB_DISABLE_XET="1")
    r = subprocess.run([sys.executable, "-c", _SCRIPT_WHISPER, str(chemin_audio)],
                       capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
    for ligne in r.stdout.splitlines():
        if ligne.startswith("@@TEXTE@@"):
            return json.loads(ligne[len("@@TEXTE@@"):])
    fin_erreur = "\n    ".join(r.stderr.strip().splitlines()[-6:])   # fin de la trace : la vraie cause
    print(f"  Whisper local indisponible :\n    {fin_erreur}")
    return None

def transcrire(chemin_audio):
    """Transcrit un fichier audio et renvoie le texte reconnu."""
    # Etape 3 : Whisper via l'API OpenAI si elle repond, sinon faster-whisper en local
    global _api_utilisable
    # Adaptation d'integration : appel facture seulement si le carnet l'autorise (run_cloud_comparison)
    if _api_utilisable and run_cloud_comparison and os.environ.get("OPENAI_API_KEY"):
        try:
            from openai import OpenAI
            client = OpenAI()
            with open(chemin_audio, "rb") as f:
                reponse = client.audio.transcriptions.create(
                    model="whisper-1", file=f, language="en"  # langue imposee : voir indices
                )
            return reponse.text
        except Exception as e:
            _api_utilisable = False   # inutile de re-essayer pour chaque fichier
            print(f"  API Whisper indisponible ({type(e).__name__}) : passage au mode local pour la suite")
    return _transcrire_local(chemin_audio)

def synthetiser_vers_fichier(texte, vitesse, chemin, voix="af_heart"):
    """Synthetise texte a la vitesse donnee, ecrit le WAV (sauf s'il existe) et renvoie sa duree en s."""
    if not chemin.exists():
        samples, sr = kokoro.create(texte, voice=voix, speed=vitesse)
        sf.write(str(chemin), samples, sr)
    return sf.info(str(chemin)).duration

mesures_intelligibilite = {}
transcriptions = {}   # (nom du texte, vitesse) -> transcription brute de Whisper
for vitesse in speed_result["results"]:
    chemin = OUTPUT_DIR / f"speed_{vitesse}_af_heart.wav"
    transcription = transcrire(chemin)
    if transcription is None:
        continue
    transcriptions[("facile", vitesse)] = transcription
    mesures_intelligibilite[vitesse] = wer(normaliser(test_text), normaliser(transcription))

if mesures_intelligibilite:
    # Etape 4 : vitesses 1.75 et 2.0 (borne haute de kokoro.create), puis un texte difficile
    # (noms propres, sigles, nombres).
    texte_difficile = (
        "Doctor Nguyen and Professor Kowalczyk met NASA and ESA engineers in Zurich "
        "on March third, twenty twenty four, to compare seventeen sensors."
    )
    vitesses = sorted(set(speed_result["results"]) | {1.75, 2.0})
    textes = {"facile": (test_text, "speed"), "difficile": (texte_difficile, "hard")}

    tableau = {nom: {} for nom in textes}   # tableau[nom][vitesse] = {"duree", "wer"}
    for nom, (texte, prefixe) in textes.items():
        reference = normaliser(texte)
        for vitesse in vitesses:
            chemin = OUTPUT_DIR / f"{prefixe}_{vitesse}_af_heart.wav"
            duree = synthetiser_vers_fichier(texte, vitesse, chemin)
            if (nom, vitesse) not in transcriptions:
                if nom == "facile" and vitesse in mesures_intelligibilite:
                    pass                                    # deja mesure plus haut
                else:
                    transcriptions[(nom, vitesse)] = transcrire(chemin) or ""
            if nom == "facile" and vitesse in mesures_intelligibilite:
                taux = mesures_intelligibilite[vitesse]
            else:
                taux = wer(reference, normaliser(transcriptions[(nom, vitesse)]))
            tableau[nom][vitesse] = {"duree": duree, "wer": taux}

    # PLANCHER DE WER : certaines erreurs ne dependent pas de la vitesse (un nom propre mal
    # orthographie l'est des 1.0). On mesure donc la degradation par rapport a la vitesse
    # normale (1.0) : c'est le SURCOUT de WER du a la vitesse qui est compare au seuil.
    plancher = {nom: tableau[nom][1.0]["wer"] for nom in textes}

    # Etape 5 : tableau vitesse / duree / WER, puis vitesse retenue
    print(f"{'Vitesse':<9}{'Duree facile':<14}{'WER facile':<12}{'Duree difficile':<17}{'WER difficile':<14}")
    print("-" * 66)
    for vitesse in vitesses:
        f, h = tableau["facile"][vitesse], tableau["difficile"][vitesse]
        marque_f = "*" if f["wer"] - plancher["facile"] > SEUIL_WER else " "
        marque_h = "*" if h["wer"] - plancher["difficile"] > SEUIL_WER else " "
        print(f"{vitesse:<9}{f['duree']:<14.2f}{f['wer']:<11.1%}{marque_f}{h['duree']:<17.2f}{h['wer']:<13.1%}{marque_h}")
    print(f"(* = surcout de WER > {SEUIL_WER:.0%} par rapport a la vitesse 1.0)")
    print(f"Plancher de WER a 1.0 : facile {plancher['facile']:.1%}, difficile {plancher['difficile']:.1%}")

    def vitesse_retenue(nom, seuil=SEUIL_WER):
        """Vitesse la plus rapide atteinte avant le PREMIER depassement du seuil de surcout.
        On s'arrete au premier depassement : un WER qui repasse sous le seuil a une
        vitesse plus elevee serait une fluctuation, pas une parole redevenue intelligible."""
        retenue, premier_rejet = None, None
        for vitesse in sorted(tableau[nom]):
            surcout = tableau[nom][vitesse]["wer"] - plancher[nom]
            if surcout <= seuil:
                if premier_rejet is None:
                    retenue = vitesse
            elif premier_rejet is None:
                premier_rejet = vitesse
        return retenue, premier_rejet

    print()
    retenues = {}
    for nom in textes:
        retenues[nom], rejet = vitesse_retenue(nom)
        if rejet is None:
            print(f"Texte {nom} : intelligible jusqu'a la vitesse maximale testee ({retenues[nom]})")
        else:
            print(f"Texte {nom} : vitesse retenue = {retenues[nom]}, seuil depasse des {rejet}")

    # Mots en erreur : permet de voir D'OU vient le plancher et ce que la vitesse fait perdre
    import difflib
    def ecarts(nom, vitesse):
        ref, hyp = normaliser(textes[nom][0]), normaliser(transcriptions.get((nom, vitesse), ""))
        ops = difflib.SequenceMatcher(a=ref, b=hyp, autojunk=False).get_opcodes()
        return [f"{' '.join(ref[i1:i2]) or '(rien)'} -> {' '.join(hyp[j1:j2]) or '(rien)'}"
                for tag, i1, i2, j1, j2 in ops if tag != "equal"]

    print()
    for nom, vitesse in [("difficile", 1.0), ("facile", max(vitesses)), ("difficile", max(vitesses))]:
        print(f"Ecarts texte {nom} a {vitesse} : {ecarts(nom, vitesse) or 'aucun'}")

    f_ret, h_ret = retenues["facile"], retenues["difficile"]
    print()
    if f_ret is not None and h_ret is not None:
        if h_ret < f_ret:
            print(f"-> Le texte difficile se degrade plus tot ({h_ret} contre {f_ret}).")
        elif h_ret == f_ret:
            print("-> Les deux textes se degradent a la meme vitesse sur cet essai.")
        else:
            print(f"-> Sur cet essai, le texte difficile tient plus longtemps ({h_ret} contre {f_ret}) : "
                  f"son plancher de WER ({plancher['difficile']:.0%}) vient d'erreurs independantes de la vitesse. "
                  "Avec une seule phrase, un mot de plus ou de moins pese 4 a 6 points : a nuancer.")
else:
    print("Aucune transcription obtenue : ni l'API Whisper ni Whisper local n'ont repondu")
    print(f"Fichiers a transcrire : {len(speed_result['results'])} (vitesses {list(speed_result['results'])})")

# Adaptation d'integration : restauration des noms partages avec l'exemple guide 2
for _nom, _valeur in _etat_avant_2b.items():
    globals()[_nom] = _valeur
del _etat_avant_2b, _nom, _valeur

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


... (5 outputs left out)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


Vitesse  Duree facile  WER facile  Duree difficile  WER difficile 
------------------------------------------------------------------
0.5      12.25         0.0%        19.09            27.3%         
0.75     8.38          0.0%        12.78            22.7%         
1.0      5.93          0.0%        9.22             22.7%         
1.25     5.06          0.0%        7.96             22.7%         
1.5      3.61          17.6%      *5.82             31.8%         
1.75     3.33          29.4%      *5.46             36.4%        *
2.0      3.05          29.4%      *4.97             40.9%        *
(* = surcout de WER > 10% par rapport a la vitesse 1.0)
Plancher de WER a 1.0 : facile 0.0%, difficile 22.7%

Texte facile : vitesse retenue = 1.25, seuil depasse des 1.5
Texte difficile : vitesse retenue = 1.5, seuil depasse des 1.75

Ecarts texte difficile a 1.0 : ['third twenty twenty four -> 3 2024', 'seventeen -> 17']
Ecarts texte facile a 2.0 : ['the -> (rien)', 'lazy -> lay', 'letter -> 

### Lecture du résultat — exemple guidé 2b

Cette exécution a appelé l'API (`run_cloud_comparison = True`, 14 transcriptions `whisper-1` réussies, une par fichier) : les WER ci-dessus sont ceux de `whisper-1`, pas de Whisper small local.

1. **Texte facile : même verdict que l'exemple 2.** Plancher nul à 1.0, vitesse retenue 1.25, premier dépassement à 1.5 (17,6 %, trois mots sur 17), comme avec Whisper small local. À 2.0, les écarts (« letter » → « layer », « alphabet » → « op ed ») sont des confusions d'écoute sur un débit trop rapide.
2. **Texte difficile : le plancher n'est fait que de nombres.** À 1.0, les noms propres sont reconnus ; les 22,7 % viennent de cinq mots : « third twenty twenty four » transcrit « 3 2024 » et « seventeen » transcrit « 17 ». Le groupe avait prévu ce cas avec `num2words`, absent de l'environnement du carnet : la normalisation ne développe pas les chiffres, et la convention d'écriture que le texte en toutes lettres voulait neutraliser est comptée comme une erreur. Ce plancher ne mesure donc pas la synthèse.
3. **Le verdict 1.5 tient à un mot.** À 1.5, le surcoût est de 9,1 points, soit deux mots sur 22, juste sous le seuil de 10 points : un mot de plus et le verdict redevenait 1.25. À 2.0, ce sont les noms propres qui cèdent (« doctor nguyen » → « knyan », « professor kowalczyk » → « przakalsic »).
4. **Comparer avec l'exemple 2 demande de la prudence.** L'exemple 2 retient 1.25 pour son propre texte difficile (date et montant en chiffres, 20 mots), avec Whisper small local et un seuil appliqué au WER brut ; l'exemple 2b retient 1.5 pour un autre texte, avec `whisper-1` et un seuil appliqué au surcoût. Trois choses changent à la fois : l'écart ne peut être attribué à aucune d'elles en particulier. L'exercice 1bis isole le reconnaisseur.

Limites : une phrase par texte, une transcription par audio ; une erreur pèse 4,5 points de WER sur le texte difficile et 5,9 points sur le texte facile.

### Exercice 1bis - Le verdict dépend-il du reconnaisseur ?

Les exemples guidés 2 et 2b mesurent l'intelligibilité avec deux reconnaisseurs : Whisper small en local (exemple 2) et, quand le carnet autorise les appels facturés, l'API Whisper d'OpenAI (exemple 2b). Le WER mesure donc la synthèse **et** le reconnaisseur : un mot mal transcrit peut venir d'une voix confuse comme d'une oreille faible. Si la vitesse retenue change avec le reconnaisseur, le verdict « intelligible jusqu'à la vitesse v » ne porte pas seulement sur Kokoro.

**Étapes** :
1. Retranscrire chaque fichier `hard_<v>_af_heart.wav` de l'exemple guidé 2b avec la fonction `transcrire()` de l'exemple guidé 2 (Whisper small local), pour les vitesses de `tableau["difficile"]` — remplir `transcriptions_locales` (dict vitesse → texte)
2. Mesurer le WER des deux reconnaisseurs sur la même référence `texte_difficile`, avec la **même** normalisation (`normaliser` et `wer` de l'exemple guidé 2) ; les transcriptions de l'exemple 2b sont dans `transcriptions[("difficile", v)]` — remplir `wer_par_reconnaisseur` (dict `"2b"` / `"local"` → dict vitesse → WER)
3. Appliquer à chaque reconnaisseur la règle du surcoût de l'exemple 2b (plancher à 1.0, `SEUIL_WER` appliqué au surcoût, arrêt au premier dépassement) — remplir `verdicts` (dict reconnaisseur → vitesse retenue). Les deux verdicts coïncident-ils ? Les deux planchers ? Quels mots expliquent l'écart ?

**Indices** :
- `# Indice` : `run_cloud_comparison` et `_api_utilisable` disent si l'exemple 2b a vraiment appelé l'API ; sinon ses transcriptions viennent aussi de Whisper local et la comparaison porte sur deux décodages du même modèle — dites-le dans la conclusion
- `# Indice` : la référence compte 22 mots, une erreur pèse donc 4,5 points de WER ; ne concluez pas sur un écart d'un seul mot

In [16]:
# Exercice 1bis : le verdict de vitesse depend-il du reconnaisseur ?
# TODO etudiant : retranscrire les hard_<v>_af_heart.wav avec Whisper local et comparer a l'exemple 2b
# Etape 1 : transcrire() de l'exemple guide 2 sur chaque hard_<v>_af_heart.wav -> transcriptions_locales
# Etape 2 : normaliser() + wer() de l'exemple guide 2 pour les deux -> wer_par_reconnaisseur
# Etape 3 : regle du surcout (plancher a 1.0, SEUIL_WER) -> verdicts (dict reconnaisseur -> vitesse)
# Indice : run_cloud_comparison et _api_utilisable disent si l'exemple 2b a vraiment appele l'API
transcriptions_locales = None  # TODO etudiant
wer_par_reconnaisseur = None  # TODO etudiant
verdicts = None  # TODO etudiant
print("Exercice 1bis a completer : le verdict depend-il du reconnaisseur ?")

Exercice 1bis a completer : le verdict depend-il du reconnaisseur ?


## Section 3 : Voix et langues

Kokoro propose plusieurs voix organisees par langue :

| Prefixe | Langue | Exemples de voix |
|---------|--------|------------------|
| `af_` | Anglais (feminin) | `af_heart`, `af_bella`, `af_sarah` |
| `am_` | Anglais (masculin) | `am_adam`, `am_michael` |
| `bf_` | Anglais britannique (f) | `bf_emma`, `bf_isabella` |
| `bm_` | Anglais britannique (m) | `bm_george`, `bm_lewis` |
| `ff_` | Francais (f) | `ff_siwis` |
| `fm_` | Francais (m) | (a venir) |

### Introduction : Exploration des voix

Kokoro propose une bibliotheque de voix pre-entraines couvrant plusieurs langues et profils vocaux.

**Dans cette section**, nous allons :
1. Tester différentes voix anglophones (af_, am_, bf_, bm_)
2. Explorer les voix francophones (ff_)
3. Mesurer les différences de performance entre voix
4. Comparer les qualites perceptuelles

Cette experimentation permet de sélectionner la voix adequate pour chaque scénario : narration, assistant virtuel, personnages, accessibilite.

In [17]:
# Test de differentes voix
print("TEST DES VOIX")
print("=" * 45)

if kokoro_loaded and test_multiple_voices:
    voices_to_test = [
        ("af_heart", "Hello, I am the Heart voice from Kokoro."),
        ("af_bella", "Hello, I am the Bella voice from Kokoro."),
        ("am_adam", "Hello, I am the Adam voice from Kokoro."),
        ("ff_siwis", "Bonjour, je suis la voix Siwis de Kokoro."),
    ]

    voice_results = {}

    for v, text in voices_to_test:
        print(f"\nVoix : {v}")
        try:
            start_time = time.time()

            if use_onnx:
                samples, sample_rate = kokoro.create(text, voice=v, speed=1.0, lang="fr-fr" if v.startswith("ff_") else "en-us")
            else:
                generator = pipeline(text, voice=v, speed=1.0, lang="fr-fr" if v.startswith("ff_") else "en-us")
                all_samples = []
                sample_rate = 24000
                for gs, ps, audio_chunk in generator:
                    all_samples.append(audio_chunk)
                samples = np.concatenate(all_samples) if all_samples else np.array([])

            gen_time = time.time() - start_time
            duration = len(samples) / sample_rate

            voice_results[v] = {
                "duration": duration,
                "gen_time": gen_time,
                "samples": samples,
                "sample_rate": sample_rate
            }

            print(f"  Duree : {duration:.1f}s | Temps : {gen_time:.2f}s | Ratio : {duration/gen_time:.1f}x")
            display_audio_array(samples, sample_rate)

            if save_results:
                filepath = OUTPUT_DIR / f"voice_{v}.wav"
                sf.write(str(filepath), samples, sample_rate)

        except Exception as e:
            print(f"  Erreur : {str(e)[:80]}")

    # Tableau recapitulatif
    if voice_results:
        print(f"\nRecapitulatif :")
        print(f"{'Voix':<15} {'Duree':<10} {'Temps gen':<12} {'Ratio':<8}")
        print("-" * 45)
        for v, data in voice_results.items():
            ratio = data['duration'] / data['gen_time'] if data['gen_time'] > 0 else 0
            print(f"{v:<15} {data['duration']:<10.1f} {data['gen_time']:<12.2f} {ratio:<8.1f}")
else:
    print("Test des voix desactive ou modele non charge")

TEST DES VOIX

Voix : af_heart


  Duree : 2.3s | Temps : 0.71s | Ratio : 3.3x


<audio embedded (audio/wav, 112684 bytes)>


Voix : af_bella


  Duree : 2.5s | Temps : 0.73s | Ratio : 3.4x


<audio embedded (audio/wav, 119852 bytes)>

... (3 outputs left out)


Voix : ff_siwis


  Duree : 2.3s | Temps : 0.60s | Ratio : 3.9x


<audio embedded (audio/wav, 110636 bytes)>


Recapitulatif :
Voix            Duree      Temps gen    Ratio   
---------------------------------------------
af_heart        2.3        0.71         3.3     
af_bella        2.5        0.73         3.4     
am_adam         2.4        0.70         3.4     
ff_siwis        2.3        0.60         3.9     


### Interprétation du test des voix

Les quatre voix ont produit des fichiers audio. Les rapports durée audio / temps de génération sont affichés dans le récapitulatif ci-dessus. Sur le CPU de cette exécution, ils se situent entre 3.0 et 3.3 : la génération est environ trois fois plus rapide que la lecture.

Les textes diffèrent entre les voix et chaque mesure ne comporte qu'un appel. Ces résultats ne permettent donc ni d'attribuer les écarts à la seule voix, ni de conclure que leur qualité est identique. Le timbre, la naturalité et la prosodie sont appréciés dans la grille d'écoute en fin de notebook (ligne « Timbre et naturel »).


## Section 4 : Comparaison avec OpenAI TTS

Comparaison entre Kokoro (local) et OpenAI TTS (cloud, payant) sur **le même texte**, dans la même session. Cette section est facultative : sans appel réel, les lignes OpenAI affichent « non mesuré ».

### Critères d'évaluation

| Critère | Description |
|---|---|
| Temps jusqu'au premier son | Délai entre l'envoi de la requête et la réception du premier fragment audio (streaming OpenAI). Pour Kokoro, génération non progressive : égal au temps total |
| Temps total | Délai jusqu'à la réception ou la génération complète de l'audio (médiane de `cloud_repetitions` appels) |
| Durée audio | Durée du son produit |
| Coût estimé | Nombre de caractères × tarif public par caractère |
| Naturalité, prosodie | Écoute des fichiers de `comparaison_cloud/`, dans la grille finale |


In [18]:
# Section 4 : comparaison Kokoro / OpenAI TTS sur le même texte, mesures réelles uniquement
import statistics, csv
print("COMPARAISON KOKORO VS OPENAI TTS")
print("=" * 45)

# Texte accentué (leçon de l'exercice 2), identique pour tous les services
comparison_text = ("L'intelligence artificielle générative transforme la façon "
                   "dont nous créons et consommons du contenu audio.")
# Tarifs publics OpenAI en dollars par million de caractères, consultés le 15/10/2026
# (modèles signalés comme dépréciés par OpenAI : un appel peut échouer)
TARIFS_PAR_MILLION_CARACTERES = {"tts-1": 15.0, "tts-1-hd": 30.0}
NON_MESURE = "non mesuré"
DOSSIER_CLOUD = OUTPUT_DIR / 'comparaison_cloud'
DOSSIER_CLOUD.mkdir(exist_ok=True)
comparaison = []

# --- Kokoro : même texte, mêmes répétitions ---
temps_kokoro = []
for _ in range(cloud_repetitions):
    t0 = time.perf_counter()
    kokoro_samples, kokoro_sr = kokoro.create(comparison_text, voice="ff_siwis", speed=1.0, lang="fr-fr")
    temps_kokoro.append(time.perf_counter() - t0)
sf.write(str(DOSSIER_CLOUD / 'kokoro_ff_siwis.wav'), kokoro_samples, kokoro_sr)
comparaison.append(dict(
    service="Kokoro ff_siwis (local, CPU)", statut="mesuré",
    premier_son_s=statistics.median(temps_kokoro), total_s=statistics.median(temps_kokoro),
    duree_audio_s=len(kokoro_samples) / kokoro_sr, cout_usd=0.0,
    fichier='comparaison_cloud/kokoro_ff_siwis.wav',
    remarque=f"médiane de {cloud_repetitions} appels ; génération non progressive"))
print("\n--- Kokoro (ff_siwis, fr-fr) ---")
display_audio_array(kokoro_samples, kokoro_sr)

# --- OpenAI : uniquement si demandé ET clé disponible ---
if not run_cloud_comparison:
    raison = "run_cloud_comparison = False"
elif not openai_key_available:
    raison = "aucune clé OPENAI_API_KEY"
else:
    raison = None
    from openai import OpenAI
    client_api = OpenAI()

for modele in cloud_models:
    ligne = dict(service=f"OpenAI {modele} (voix alloy)", statut=NON_MESURE, premier_son_s=None,
                 total_s=None, duree_audio_s=None, cout_usd=None, fichier=None, remarque=raison)
    if raison is None:
        try:
            premiers, totaux = [], []
            for _ in range(cloud_repetitions):
                t0, premier, morceaux = time.perf_counter(), None, []
                with client_api.audio.speech.with_streaming_response.create(
                        model=modele, voice="alloy", input=comparison_text, response_format="pcm") as reponse:
                    for morceau in reponse.iter_bytes():
                        if premier is None:
                            premier = time.perf_counter() - t0
                        morceaux.append(morceau)
                totaux.append(time.perf_counter() - t0)
                premiers.append(premier)
            brut = b''.join(morceaux)
            # Format « pcm » de l'API : 24 kHz, 16 bits signés, mono, petit-boutiste
            pcm = np.frombuffer(brut[:len(brut) // 2 * 2], dtype='<i2').astype(np.float32) / 32768
            sr_openai = 24000
            fichier = f'openai_{modele}_alloy.wav'
            sf.write(str(DOSSIER_CLOUD / fichier), pcm, sr_openai)
            tarif = TARIFS_PAR_MILLION_CARACTERES.get(modele)
            ligne.update(statut="mesuré", premier_son_s=statistics.median(premiers),
                         total_s=statistics.median(totaux), duree_audio_s=len(pcm) / sr_openai,
                         cout_usd=len(comparison_text) * tarif / 1e6 if tarif else None,
                         fichier=f'comparaison_cloud/{fichier}',
                         remarque=f"médiane de {cloud_repetitions} appels, réseau inclus"
                                  + ("" if tarif else " ; tarif inconnu, coût non estimé"))
            print(f"\n--- OpenAI {modele} ---")
            display_audio_array(pcm, sr_openai)
        except Exception as e:
            ligne['remarque'] = f"échec de l'appel : {type(e).__name__}: {str(e)[:120]}"
    comparaison.append(ligne)

def _cellule(valeur, fmt):
    return NON_MESURE if valeur is None else format(valeur, fmt)

print(f"\n{'=' * 92}\nTABLEAU COMPARATIF (mesures de cette session)\n{'=' * 92}")
print(f"{'Service':<32} {'1er son (s)':>12} {'Total (s)':>12} {'Durée (s)':>12} {'Coût ($)':>12}")
print("-" * 92)
for l in comparaison:
    print(f"{l['service']:<32} {_cellule(l['premier_son_s'], '.2f'):>12} {_cellule(l['total_s'], '.2f'):>12} "
          f"{_cellule(l['duree_audio_s'], '.2f'):>12} {_cellule(l['cout_usd'], '.5f'):>12}")
for l in comparaison:
    if l['remarque']:
        print(f"  {l['service']} : {l['remarque']}")
with open(OUTPUT_DIR / 'comparaison_cloud.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(comparaison[0]))
    writer.writeheader()
    writer.writerows([{k: (NON_MESURE if v is None else v) for k, v in l.items()} for l in comparaison])

# Ordre de grandeur pour une heure de narration (~50 000 caractères), à partir des tarifs publics
chars_1h = 50000
print(f"\nEstimation pour 1 heure de narration (~{chars_1h:,} caractères, tarifs publics, hors matériel) :")
print("  Kokoro          : pas de facturation par requête (coût du matériel et de l'énergie)")
for modele, tarif in TARIFS_PAR_MILLION_CARACTERES.items():
    print(f"  OpenAI {modele:<9}: {chars_1h * tarif / 1e6:.2f} $")


COMPARAISON KOKORO VS OPENAI TTS


--- Kokoro (ff_siwis, fr-fr) ---


<audio embedded (audio/wav, 305196 bytes)>

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/speech "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/speech "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/speech "HTTP/1.1 200 OK"


... (4 outputs left out)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/speech "HTTP/1.1 200 OK"



--- OpenAI tts-1-hd ---


<audio embedded (audio/wav, 331844 bytes)>


TABLEAU COMPARATIF (mesures de cette session)
Service                           1er son (s)    Total (s)    Durée (s)     Coût ($)
--------------------------------------------------------------------------------------------
Kokoro ff_siwis (local, CPU)             1.48         1.48         6.36      0.00000
OpenAI tts-1 (voix alloy)                1.42         2.32         6.76      0.00161
OpenAI tts-1-hd (voix alloy)             2.36         2.85         6.91      0.00321
  Kokoro ff_siwis (local, CPU) : médiane de 3 appels ; génération non progressive
  OpenAI tts-1 (voix alloy) : médiane de 3 appels, réseau inclus
  OpenAI tts-1-hd (voix alloy) : médiane de 3 appels, réseau inclus

Estimation pour 1 heure de narration (~50,000 caractères, tarifs publics, hors matériel) :
  Kokoro          : pas de facturation par requête (coût du matériel et de l'énergie)
  OpenAI tts-1    : 0.75 $
  OpenAI tts-1-hd : 1.50 $


### Comparaison : statut et limites

- **Mesures réelles uniquement.** Les lignes OpenAI ne contiennent que des valeurs mesurées dans cette session. Sans appel (paramètre désactivé, clé absente, modèle retiré), elles affichent « non mesuré » avec la raison, et ne doivent recevoir ni interprétation ni note d'écoute.
- **Même texte.** Le texte est accentué (leçon de l'exercice 2) et identique pour tous les services. Kokoro utilise `ff_siwis` et `fr-fr` ; OpenAI utilise la voix `alloy`.
- **Temps.** Les temps OpenAI incluent le réseau. Ceux de Kokoro dépendent du CPU de la session. Ce sont des mesures de cette session, pas des performances générales. Le streaming OpenAI permet d'entendre le début avant la fin de la génération ; Kokoro, tel qu'il est appelé ici, ne rend l'audio qu'une fois complet.
- **Coûts.** Les tarifs publics (tts-1 : 15 $, tts-1-hd : 30 $ par million de caractères) ont été consultés le 15/10/2026. OpenAI signale ces modèles comme dépréciés. Pour tester un autre modèle, l'ajouter à `cloud_models` ; son coût n'est alors pas estimé.
- **Écoute.** Les fichiers de `comparaison_cloud/` permettent de noter la naturalité et la prosodie dans la grille finale, uniquement pour les services effectivement mesurés.


### Exemple guidé 3 : Pipeline de synthèse par lot

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20681), intégrée comme exemple guidé.*

Cet exemple convertit une petite liste d'articles (titre et résumé) en fichiers audio, un fichier par article, comme pour un flux de type « podcast actualités ».

**Objectif** : enchaîner les synthèses, nommer les fichiers de façon systématique et produire le bilan du lot.

**Étapes** :
1. Construire le texte de chaque article : `title + ". " + summary`
2. Le synthétiser avec `kokoro.create(...)` en chronométrant l'appel
3. Sauvegarder `article_<id>_<voix>.wav` dans `OUTPUT_DIR`
4. Cumuler durées et temps de génération, en isolant les erreurs article par article (`try/except`)
5. Renvoyer un résumé : nombre d'articles, durée totale, temps total, erreurs

**Points clés du code** :
- l'isolement des erreurs par article rend le lot robuste : un texte vide ou un caractère non géré ne coûte qu'un fichier, pas le lot entier
- les articles sont en français, mais la voix `af_heart` est anglaise et `kokoro.create()` phonémise par défaut selon les règles de l'anglais (`lang="en-us"`) : l'audio est produit sans erreur, avec une prononciation anglaise. L'exercice 2 mesure cet effet

In [19]:
def batch_tts_pipeline(articles, voice="af_heart", output_dir=None):
    """Synthetise un batch d'articles en fichiers audio."""
    total_duration = 0
    total_gen_time = 0
    errors = []

    for article in articles:
        try:
            # Texte complet = titre + resume
            text = article["title"] + ". " + article["summary"]

            # Generer l'audio
            debut = time.time()
            samples, sample_rate = kokoro.create(text, voice=voice, speed=1.0)
            gen_time = time.time() - debut

            # Sauvegarder le fichier
            filepath = output_dir / f"article_{article['id']}_{voice}.wav"
            sf.write(str(filepath), samples, sample_rate)

            # Metriques
            duration = len(samples) / sample_rate
            total_duration += duration
            total_gen_time += gen_time
            print(f"Article {article['id']} : {duration:.1f}s d'audio en {gen_time:.2f}s (ratio {duration / gen_time:.1f}x)")
        except Exception as e:
            errors.append((article["id"], str(e)))
            print(f"Article {article['id']} : erreur {e}")

    return {
        "nb_articles": len(articles),
        "total_duration": total_duration,
        "total_gen_time": total_gen_time,
        "errors": errors
    }

# Donnees de test
test_articles = [
    {"id": 1, "title": "IA et sante", "summary": "L'intelligence artificielle revolutionne le diagnostic medical."
},
    {"id": 2, "title": "Voitures autonomes", "summary": "Les vehicules autonomes progressent rapidement sur les routes europeennes."
},
    {"id": 3, "title": "Quantique", "summary": "Les ordinateurs quantiques promettent des avancees majeures en cryptographie."
},
]

# Test du pipeline
output_dir = OUTPUT_DIR
result = batch_tts_pipeline(test_articles, voice="af_heart", output_dir=OUTPUT_DIR)
print(f"\nResume : {result['nb_articles']} articles, {result['total_duration']:.1f}s d'audio, "
      f"{result['total_gen_time']:.2f}s de generation, {len(result['errors'])} erreur(s)")

Article 1 : 5.4s d'audio en 1.31s (ratio 4.1x)


Article 2 : 6.1s d'audio en 1.45s (ratio 4.3x)


Article 3 : 5.8s d'audio en 1.41s (ratio 4.1x)

Resume : 3 articles, 17.3s d'audio, 4.17s de generation, 0 erreur(s)


### Exemple guidé 4 (crédité #21036) : Phonémisation — la bonne langue et les accents

Avant de générer un son, Kokoro convertit le texte en **phonèmes** avec espeak-ng, selon la langue passée dans le paramètre `lang` de `kokoro.create()` (par défaut `"en-us"`). L'exemple guidé 3 a donc fait lire des résumés français avec des règles de prononciation anglaises. Le défaut ne lève aucune erreur : il s'entend à l'écoute, et il se lit dans les phonèmes.

**Contexte** : les textes de l'exemple sont écrits sans accents (`sante`, `vehicules`, `europeennes`). Même en `"fr-fr"`, cela produit deux défauts de nature différente :
- un mot absent du dictionnaire français est traité comme un mot étranger : espeak-ng **bascule de langue** en plein mot, ce que la chaîne de phonèmes signale par des marqueurs `(en)` … `(fr)` ;
- d'autres mots restent en français mais sont **mal prononcés sans aucun marqueur** : `sante` devient « sant », `medical` perd son « é ».

**Étapes** :
1. Pour le texte de chaque article de l'exemple guidé 3, afficher les phonèmes produits par `kokoro.tokenizer.phonemize(texte, "en-us")` puis par `"fr-fr"`, et les comparer
2. Compter dans la version `"fr-fr"` les bascules vers l'anglais (marqueurs `(en)`)
3. Réécrire les textes avec leurs accents (`santé`, `révolutionne`, `véhicules`, `européennes`, `avancées`…), puis comparer leurs phonèmes à ceux de l'étape 1 : les bascules disparaissent-elles, et quels mots changent de prononciation sans en avoir signalé ?
4. Le sigle « IA » est lu comme un mot d'une syllabe : trouver une écriture qui le fait épeler, et vérifier dans les phonèmes
5. Relancer la synthèse du lot avec la voix française `ff_siwis` et `lang="fr-fr"` sur les textes corrigés, puis écrire un manifeste `episode.json` qui décrit l'épisode : pour chaque article, le fichier, la voix, la langue, la durée et l'instant de début dans l'épisode concaténé (avec une pause d'une seconde entre deux articles)

**Indices** :
- `# Indice` : `kokoro.tokenizer.phonemize(texte, lang)` renvoie une chaîne ; `chaine.count("(en)")` compte les bascules
- `# Indice` : `kokoro.get_voices()` liste les voix disponibles ; `ff_siwis` est la seule voix française
- `# Indice` : l'instant de début d'un article est la somme des durées des articles précédents et des pauses qui les séparent

> **Exemple guidé (crédité #21036).** La résolution ci-dessous est celle du groupe de la PR #21036, conservée telle quelle : comptage des bascules `(en)` dans les phonèmes `fr-fr` de textes sans accents, réécriture accentuée, et comparaison phonème par phonème. Le nouveau **Exercice 2** qui suit utilise le même moteur de phonémisation dans l'autre sens : le nombre de phonèmes prédit-il la durée de parole ?


In [20]:
# Exemple guidé 4 : phonèmes complets, accents réels, langue explicite et manifeste
def compter_bascules(phonemes):
    return phonemes.count('(en)')

articles_corriges = [
    {'id': 1, 'title': 'I.A. et santé', 'summary': "L'intelligence artificielle révolutionne le diagnostic médical."},
    {'id': 2, 'title': 'Voitures autonomes', 'summary': 'Les véhicules autonomes progressent rapidement sur les routes européennes.'},
    {'id': 3, 'title': 'Quantique', 'summary': 'Les ordinateurs quantiques promettent des avancées majeures en cryptographie.'}
]
analyses = []
for original, corrige in zip(test_articles, articles_corriges):
    brut = original['title'] + '. ' + original['summary']
    propre = corrige['title'] + '. ' + corrige['summary']
    en = kokoro.tokenizer.phonemize(brut, 'en-us')
    fr = kokoro.tokenizer.phonemize(brut, 'fr-fr')
    fr_corr = kokoro.tokenizer.phonemize(propre, 'fr-fr')
    print('\nArticle', original['id'], '\nEN :', en, '\nFR :', fr, '\nFR corrigé :', fr_corr)
    print('Bascules avant / après :', compter_bascules(fr), compter_bascules(fr_corr))
    analyses.append(dict(id=original['id'], en=en, fr=fr, fr_corrige=fr_corr))
print('\nComparaison mot par mot :')
for avant, apres in [('sante','santé'), ('revolutionne','révolutionne'), ('medical','médical'),
                     ('vehicules','véhicules'), ('europeennes','européennes'), ('avancees','avancées')]:
    p, q = [kokoro.tokenizer.phonemize(m, 'fr-fr') for m in (avant, apres)]
    print(avant, p, '->', apres, q, '| changement sans bascule signalée :', p != q and '(en)' not in p)
print('\nÉpellation :')
for variante in ['IA', 'I-A', 'I.A.', 'I A']:
    print(variante, '->', kokoro.tokenizer.phonemize(variante, 'fr-fr'))
print('Les marqueurs ne détectent que les bascules de langue ; leur absence ne garantit pas '
      'une prononciation correcte. Les comparaisons ci-dessus isolent les changements sans bascule.')

manifeste = {'title': 'Actualité technologique', 'articles': [], 'pause_seconds': 1.0}
segments_episode, position, episode_sr = [], 0, None
for i, article in enumerate(articles_corriges):
    texte = article['title'] + '. ' + article['summary']
    audio, sr = kokoro.create(texte, voice='ff_siwis', speed=1.0, lang='fr-fr')
    if episode_sr is None: episode_sr = sr
    assert sr == episode_sr
    fichier = f"article_{article['id']}_ff_siwis.wav"
    sf.write(str(OUTPUT_DIR / fichier), audio, sr)
    manifeste['articles'].append(dict(id=article['id'], file=fichier, voice='ff_siwis',
        lang='fr-fr', text=texte, duration_seconds=len(audio)/sr, start_time_seconds=position/sr,
        start_sample=position, num_samples=len(audio)))
    segments_episode.append(audio); position += len(audio)
    if i < len(articles_corriges)-1:
        pause = np.zeros(sr, dtype=np.float32)
        segments_episode.append(pause); position += len(pause)
full_podcast = np.concatenate(segments_episode)
assert len(full_podcast) == position
manifeste.update(sample_rate=episode_sr, duration_seconds=position/episode_sr, file='podcast_complet_fr.wav')
sf.write(str(OUTPUT_DIR / manifeste['file']), full_podcast, episode_sr)
(OUTPUT_DIR / 'episode.json').write_text(json.dumps(manifeste, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'phonemes.json').write_text(json.dumps(analyses, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifeste, ensure_ascii=False, indent=2))
display_audio_array(full_podcast, episode_sr)


Article 1 
EN : ˌaɪˈeɪ ˈɛt sˈænt. ˈɛlˈɪntɪlˌɪdʒəns ˌɑːɹɾɪfˌɪsɪˈɛl ɹˌɛvəlˈuːʃən lə dˌaɪəɡnˈɑːstɪk mˈɛdɪkəl. 
FR : jˈa e sˈɑ̃t. lɛ̃tɛliʒˈɑ̃s aʁtifisjˈɛl ʁəvolysjˈɔn lə djaɡnɔstˈik mədikˈal. 
FR corrigé : ˈi.ˈa. e sɑ̃tˈe. lɛ̃tɛliʒˈɑ̃s aʁtifisjˈɛl ʁevolysjˈɔn lə djaɡnɔstˈik medikˈal.
Bascules avant / après : 0 0

Article 2 
EN : vˈɔɪtʃɚz ɔːtˈɑːnoʊmz. lˈɛs vˈɛhɪkjˌuːlz ɔːtˈɑːnoʊmz pɹəɡɹɪsˈɛnt ɹˈæpaɪdmənt sˈɜː lˈɛs ɹˈaʊts jˈʊɹɹəpˌiːnz. 
FR : vwatˈyʁz otonˈom. le vəikˈylz otonˈom pʁɔɡʁˈɛs ʁapidmˈɑ̃ syʁ le ʁˈut (en)jˈʊəɹəpˌiːnz(fr). 
FR corrigé : vwatˈyʁz otonˈom. le veikˈylz otonˈom pʁɔɡʁˈɛs ʁapidmˈɑ̃ syʁ le ʁˈutz øʁopeˈɛn.
Bascules avant / après : 1 0

Article 3 
EN : kwɔntˈiːk. lˈɛs ˈɔːɹdᵻnɐtˈʊɹz kwɔntˈiːks pɹˈoʊmtənt dˈɛs ˈævɐnsˌiːz mɐdʒjˈuːɹəz ˈɛn kɹˈɪptəɡɹˌæfi. 
FR : kɑ̃tˈik. lez ɔʁdinatˈœʁ kɑ̃tˈik pʁomˈɛt de (en)ˈavansˌiːz(fr) maʒˈœʁz ɑ̃ kʁiptɔɡʁafˈi. 
FR corrigé : kɑ̃tˈik. lez ɔʁdinatˈœʁ kɑ̃tˈik pʁomˈɛt dez avɑ̃sˈe maʒˈœʁz ɑ̃ kʁiptɔɡʁafˈi.
Bascules avant / après : 1 0

Comparaison mot

{
  "title": "Actualité technologique",
  "articles": [
    {
      "id": 1,
      "file": "article_1_ff_siwis.wav",
      "voice": "ff_siwis",
      "lang": "fr-fr",
      "text": "I.A. et santé. L'intelligence artificielle révolutionne le diagnostic médical.",
      "duration_seconds": 4.928,
      "start_time_seconds": 0.0,
      "start_sample": 0,
      "num_samples": 118272
    },
    {
      "id": 2,
      "file": "article_2_ff_siwis.wav",
      "voice": "ff_siwis",
      "lang": "fr-fr",
      "text": "Voitures autonomes. Les véhicules autonomes progressent rapidement sur les routes européennes.",
      "duration_seconds": 5.056,
      "start_time_seconds": 5.928,
      "start_sample": 142272,
      "num_samples": 121344
    },
    {
      "id": 3,
      "file": "article_3_ff_siwis.wav",
      "voice": "ff_siwis",
      "lang": "fr-fr",
      "text": "Quantique. Les ordinateurs quantiques promettent des avancées majeures en cryptographie.",
      "duration_seconds": 4.672,
     

<audio embedded (audio/wav, 799532 bytes)>

### Exemple guidé 4 (prolongement) : vérifications reproductibles et plan factoriel

**1. Le sigle à la synthèse.** La cellule synthétise « IA et santé. » avec quatre écritures du sigle et mesure, pour chacune, la durée et le plus long silence interne. Cela vérifie que les points de « I.A. » ne créent pas de pause gênante. Résultats : `sigle_ia.csv`.

**2. Un facteur à la fois.** Entre l'exemple guidé 3 et l'exercice 2, trois facteurs changeaient en même temps : la voix (`af_heart` → `ff_siwis`), la langue de phonémisation (`en-us` → `fr-fr`) et l'orthographe (sans → avec accents). Le plan factoriel synthétise les trois articles dans **les huit combinaisons**. Toutes sont techniquement possibles avec Kokoro, mais une voix utilisée avec la phonémisation d'une autre langue sort de ses conditions d'entraînement : ces combinaisons croisées servent à isoler un effet, pas à produire un audio de qualité.

Chaque audio est transcrit par Whisper en français (`language="fr"`, `beam_size=5`). Le WER est calculé contre le texte accentué, sans tenir compte des accents ni de l'écriture du sigle (« I.A. » et « IA » sont équivalents). Pour chaque facteur, la cellule compare les quatre paires de conditions qui ne diffèrent **que par ce facteur**. Résultats : `plan_factoriel.csv` ; audios dans `plan_factoriel/`.


In [21]:
# Exemple guidé 4 (prolongement) : sigle à la synthèse, puis plan factoriel voix x langue x orthographe
import itertools, statistics

def normaliser_fr(texte):
    """normaliser() sans accents, avec les sigles à points regroupés (« I.A. » -> « ia »)."""
    texte = re.sub(r'\b(?:\w\.){2,}', lambda m: m.group(0).replace('.', ''), texte)
    texte = ''.join(c for c in unicodedata.normalize('NFD', texte) if unicodedata.category(c) != 'Mn')
    return normaliser(texte)

def plus_long_silence_ms(audio, sr, seuil_db=-40, fenetre_ms=10):
    """Plus long silence entre le début et la fin de la parole (fenêtres de 10 ms sous -40 dBFS)."""
    n = int(sr * fenetre_ms / 1000); k = len(audio) // n
    rms = np.sqrt((audio[:k * n].reshape(k, n) ** 2).mean(axis=1))
    parole = np.where(rms >= 10 ** (seuil_db / 20))[0]
    if len(parole) == 0:
        return 0
    meilleur = courant = 0
    for silencieux in rms[parole[0]:parole[-1] + 1] < 10 ** (seuil_db / 20):
        courant = courant + 1 if silencieux else 0
        meilleur = max(meilleur, courant)
    return meilleur * fenetre_ms

# 1. Le sigle à la synthèse
lignes_sigle = []
print("Sigle : écriture -> phonèmes, durée, plus long silence interne (ff_siwis, fr-fr)")
for ecriture in ['IA', 'I-A', 'I.A.', 'I A']:
    phrase = f'{ecriture} et santé.'
    audio, sr = kokoro.create(phrase, voice='ff_siwis', speed=1.0, lang='fr-fr')
    lignes_sigle.append(dict(ecriture=ecriture, phonemes=kokoro.tokenizer.phonemize(phrase, 'fr-fr'),
                             duree_s=len(audio) / sr, plus_long_silence_ms=plus_long_silence_ms(audio, sr)))
    l = lignes_sigle[-1]
    print(f"  {ecriture:5} -> {l['phonemes']:22} {l['duree_s']:.2f} s, silence max {l['plus_long_silence_ms']} ms")
with open(OUTPUT_DIR / 'sigle_ia.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(lignes_sigle[0]))
    writer.writeheader(); writer.writerows(lignes_sigle)

# 2. Plan factoriel 2 x 2 x 2
TEXTES_EX2 = {'sans_accents': [a['title'] + '. ' + a['summary'] for a in test_articles],
              'avec_accents': [a['title'] + '. ' + a['summary'] for a in articles_corriges]}
FACTEURS = {'voix': ['af_heart', 'ff_siwis'], 'langue': ['en-us', 'fr-fr'],
            'orthographe': ['sans_accents', 'avec_accents']}
DOSSIER_FACTORIEL = OUTPUT_DIR / 'plan_factoriel'
lignes_factoriel, effets_factoriel = [], {}

if not run_plan_factoriel:
    print("\nPlan factoriel désactivé (run_plan_factoriel = False) : non mesuré.")
else:
    DOSSIER_FACTORIEL.mkdir(exist_ok=True)
    for voix_c, langue_c, ortho_c in itertools.product(*FACTEURS.values()):
        for i, texte in enumerate(TEXTES_EX2[ortho_c]):
            audio, sr = kokoro.create(texte, voice=voix_c, speed=1.0, lang=langue_c)
            nom_fichier = f'article_{i + 1}_{voix_c}_{langue_c}_{ortho_c}.wav'
            sf.write(str(DOSSIER_FACTORIEL / nom_fichier), audio, sr)
            hyp = transcrire(DOSSIER_FACTORIEL / nom_fichier, langue='fr', beam_size=5)
            ref = normaliser_fr(TEXTES_EX2['avec_accents'][i])
            lignes_factoriel.append(dict(
                article=i + 1, voix=voix_c, langue=langue_c, orthographe=ortho_c,
                bascules_vers_anglais=kokoro.tokenizer.phonemize(texte, langue_c).count('(en)'),
                mots_reference=len(ref), erreurs=distance_mots(ref, normaliser_fr(hyp)),
                transcription=hyp, fichier=f'plan_factoriel/{nom_fichier}'))
    with open(OUTPUT_DIR / 'plan_factoriel.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(lignes_factoriel[0]))
        writer.writeheader(); writer.writerows(lignes_factoriel)

    def wer_condition(**cond):
        sel = [l for l in lignes_factoriel if all(l[k] == v for k, v in cond.items())]
        return sum(l['erreurs'] for l in sel) / sum(l['mots_reference'] for l in sel)

    print(f"\n{'voix':9} {'langue':6} {'orthographe':13} {'WER (3 articles)':>17} {'bascules':>9}")
    for voix_c, langue_c, ortho_c in itertools.product(*FACTEURS.values()):
        sel = [l for l in lignes_factoriel if (l['voix'], l['langue'], l['orthographe']) == (voix_c, langue_c, ortho_c)]
        print(f"{voix_c:9} {langue_c:6} {ortho_c:13} {wer_condition(voix=voix_c, langue=langue_c, orthographe=ortho_c):17.1%} "
              f"{sum(l['bascules_vers_anglais'] for l in sel):9d}")

    print("\nEffet d'un seul facteur, les deux autres étant fixés (WER du 1er niveau -> WER du 2e niveau) :")
    for facteur, (niveau_1, niveau_2) in FACTEURS.items():
        autres = [f for f in FACTEURS if f != facteur]
        ecarts_facteur = []
        for combinaison in itertools.product(*(FACTEURS[f] for f in autres)):
            fixe = dict(zip(autres, combinaison))
            w1, w2 = wer_condition(**fixe, **{facteur: niveau_1}), wer_condition(**fixe, **{facteur: niveau_2})
            ecarts_facteur.append(w2 - w1)
            print(f"  {facteur:11} {niveau_1} -> {niveau_2} | {', '.join(f'{k}={v}' for k, v in fixe.items()):38} | "
                  f"{w1:6.1%} -> {w2:6.1%}")
        effets_factoriel[facteur] = statistics.mean(ecarts_facteur)
        print(f"  => écart moyen dû à « {facteur} » : {effets_factoriel[facteur]:+.1%}\n")


Sigle : écriture -> phonèmes, durée, plus long silence interne (ff_siwis, fr-fr)


  IA    -> jˈa e sɑ̃tˈe.          1.02 s, silence max 50 ms


  I-A   -> ja e sɑ̃tˈe.           1.00 s, silence max 50 ms


  I.A.  -> ˈi.ˈa. e sɑ̃tˈe.       1.11 s, silence max 70 ms


  I A   -> ˈi a e sɑ̃tˈe.         1.05 s, silence max 50 ms



voix      langue orthographe    WER (3 articles)  bascules
af_heart  en-us  sans_accents              30.0%         0
af_heart  en-us  avec_accents              30.0%         0
af_heart  fr-fr  sans_accents              63.3%         2
af_heart  fr-fr  avec_accents              23.3%         0
ff_siwis  en-us  sans_accents              53.3%         0
ff_siwis  en-us  avec_accents              26.7%         0
ff_siwis  fr-fr  sans_accents              43.3%         2
ff_siwis  fr-fr  avec_accents               0.0%         0

Effet d'un seul facteur, les deux autres étant fixés (WER du 1er niveau -> WER du 2e niveau) :
  voix        af_heart -> ff_siwis | langue=en-us, orthographe=sans_accents |  30.0% ->  53.3%
  voix        af_heart -> ff_siwis | langue=en-us, orthographe=avec_accents |  30.0% ->  26.7%
  voix        af_heart -> ff_siwis | langue=fr-fr, orthographe=sans_accents |  63.3% ->  43.3%
  voix        af_heart -> ff_siwis | langue=fr-fr, orthographe=avec_accents |  23.3% ->

### Analyse de l'exemple guidé 4

**1. Règles anglaises contre règles françaises.** Avec `en-us`, espeak-ng applique l'orthographe anglaise à des mots français : « Les » → `lˈɛs` (le *s* final se prononce), « routes » → `ɹˈaʊts` (comme l'anglais *routes*), « en » → `ˈɛn`, « medical » → `mˈɛdɪkəl`, « promettent » → `pɹˈoʊmtənt`. Le *r* est anglais (`ɹ`) et aucune voyelle nasale française (`ɑ̃`, `ɔ̃`) n'apparaît. Seul le sigle est épelé (`ˌaɪˈeɪ`), mais avec les noms anglais des lettres. Avec `fr-fr`, on retrouve le `ʁ` français, les nasales (`sˈɑ̃t`, `ʁapidmˈɑ̃`) et les finales muettes (`le`, `ʁˈut`).

**2. Bascules vers l'anglais.** Sans accents, en `fr-fr`, les trois articles en comptent respectivement **0, 1 et 1**. « europeennes » et « avancees » sont absents du dictionnaire français, si bien qu'espeak-ng les lit comme de l'anglais : `(en)jˈʊəɹəpˌiːnz(fr)` et `(en)ˈavansˌiːz(fr)`.

**3. Effet des accents sur les phonèmes.** Une fois les accents rétablis, il n'y a plus aucune bascule. Mais quatre autres mots changent de prononciation sans avoir jamais déclenché de marqueur : sans accent, le *e* est lu comme un *e* muet (`ə`) ; avec l'accent, il devient un *é* (`e`).

| Sans accent | Avec accent |
|---|---|
| sante `sˈɑ̃t` (« sant ») | santé `sɑ̃tˈe` |
| revolutionne `ʁəvolysjˈɔn` | révolutionne `ʁevolysjˈɔn` |
| medical `mədikˈal` | médical `medikˈal` |
| vehicules `vəikˈyl` | véhicules `veikˈyl` |

À retenir : l'absence de marqueur `(en)` ne garantit pas une prononciation correcte. Les textes corrigés font aussi apparaître des **liaisons** : « des‿avancées » (`dez avɑ̃sˈe`), obligatoire, et « routes‿européennes » (`ʁˈutz øʁopeˈɛn`), facultative et un peu soutenue, mais acceptable pour une lecture d'actualités.

**4. Le sigle.** Dans les phonèmes, `IA` (`jˈa`) et `I-A` (`ja`) se lisent comme un seul mot, « ya », tandis que `I.A.` (`ˈi.ˈa.`) et `I A` (`ˈi a`) sont épelés. La cellule de vérification mesure la durée et le plus long silence interne de chaque écriture ; lors de la préparation de la V4, les points de `I.A.` n'ajoutaient qu'environ 0,1 s à la phrase, sans pause marquée. `I.A.` est donc retenu.

**5. Le manifeste.** Les calculs sont justes : l'article 2 commence à 4,928 + 1 = 5,928 s, l'article 3 à 5,928 + 5,056 + 1 = 11,984 s, et l'épisode dure 11,984 + 4,672 = 16,656 s, dont 2 s de pauses. Les positions sont aussi enregistrées en échantillons (`start_sample`), ce qui évite les erreurs d'arrondi.

**6. Attribuer l'amélioration à un facteur.** La comparaison directe entre l'exemple guidé 3 (`af_heart`, `en-us`, sans accents) et l'exemple guidé 4 (`ff_siwis`, `fr-fr`, avec accents) changeait trois facteurs à la fois. Le plan factoriel les sépare. Sa sortie se lit ainsi :
- l'**écart moyen** de chaque facteur indique de combien le WER varie quand on change ce seul facteur, les deux autres restant fixés ;
- les **bascules** ne dépendent que de la langue et de l'orthographe : la voix n'intervient pas dans la phonémisation ;
- si les quatre écarts d'un facteur sont de même signe, son effet est cohérent ; s'ils changent de signe, il dépend des autres facteurs (interaction), et un écart moyen ne suffit pas à le résumer.

Ce que mesure ce WER, et ce qu'il ne mesure pas : il indique si Whisper retrouve le texte français. Il ne mesure pas l'accent : Whisper peut reconstituer un texte lu avec un fort accent anglais. Les combinaisons croisées (voix anglaise et phonèmes français, ou l'inverse) sortent des conditions d'entraînement des voix. Enfin, avec trois articles courts (environ 30 mots par condition), une erreur pèse environ 3 % : seuls des écarts nets sont interprétables.

**Écoute.** L'appréciation à l'oreille figure dans la grille finale (ligne « Prononciation française »). Pour isoler un facteur à l'écoute, comparer deux fichiers de `plan_factoriel/` qui ne diffèrent que par ce facteur, par exemple `article_2_af_heart_fr-fr_avec_accents.wav` et `article_2_ff_siwis_fr-fr_avec_accents.wav` pour l'effet de la voix.

Le détail des phonèmes est enregistré dans `phonemes.json`.


### Exercice 2 : Phonèmes et durée — le comptage prédit-il le temps de parole ?

L'exemple guidé 4 a lu les phonèmes comme un **diagnostic** de langue. Un moteur de synthèse les consomme aussi comme du **carburant** : plus une réplique porte de phonèmes, plus l'audio est long — à condition que le débit (phonèmes par seconde) reste à peu près stable d'une voix à l'autre.

**Étapes** :
1. Pour chaque réplique du dialogue de l'exemple guidé 5 (texte + voix), compter ses phonèmes via `kokoro.tokenizer.phonemize(texte, langue_de_la_voix)` — remplir `phonemes_par_replica` (liste d'entiers, dans l'ordre du dialogue)
2. Relever la durée de chaque réplique depuis les segments pairs d'`audio_segments` (l'infrastructure de l'exemple guidé 6) — remplir `durees_par_replica` (liste de secondes)
3. Calculer le débit de chaque voix : `phonèmes / durée`, regroupé par voix — remplir `debit_phonemes_par_voix` (dict voix → liste de débits). Les deux voix parlent-elles à des débits comparables ?

**Indices** :
- `# Indice` : la langue de phonémisation suit la voix (`af_heart` → `"en-us"`, voix françaises → `"fr-fr"`) ; une mauvaise langue gonfle ou dégonfle le compte
- `# Indice` : le débit se compare en rapport (max/min), pas en différence absolue — un rapport ≤ 1,2 dira « comparable »

In [22]:
# Exercice 2 : phonemes et duree -- le comptage predit-il le temps de parole ?
# TODO etudiant : compter les phonemes de chaque replique et comparer aux durees des segments
# Etape 1 : len(phonemize(texte, langue)) par replique -> phonemes_par_replica (liste d'entiers)
# Etape 2 : duree des segments pairs d'audio_segments -> durees_par_replica (liste de secondes)
# Etape 3 : debit = phonemes / duree, regroupe par voix -> debit_phonemes_par_voix (dict voix -> listes)
# Indice : la langue de phonemisation suit la voix (af_heart -> en-us, voix fr -> fr-fr)
phonemes_par_replica = None  # TODO etudiant
durees_par_replica = None  # TODO etudiant
debit_phonemes_par_voix = None  # TODO etudiant
print("Exercice 2 a completer : phonemes et duree")

Exercice 2 a completer : phonemes et duree


### Exemple guidé 4b (crédité #28524) : la même consigne, comparée en contexte

La consigne est celle de l'exemple guidé 4 ; la solution ci-dessous vient d'un **second groupe**, celui de la PR #28524, conservée quasi verbatim. L'exemple 4 compare les mots sans accents et avec accents **isolément** (`sante` → `santé`) ; celui-ci phonémise les **articles entiers**, puis compare les phonèmes position par position. Il distingue deux cas : une **bascule corrigée** (le mot était marqué `(en)` et ne l'est plus) et un changement **silencieux** (la prononciation change sans qu'aucun marqueur ne l'ait signalé). Il vérifie aussi par une assertion que l'écriture `I.A.` fait bien épeler le sigle.

Adaptations d'intégration, déclarées dans la PR : titre reformulé, et un commentaire renvoie à l'exemple guidé 3, où `test_articles` est défini dans ce carnet ; les fichiers (`article_*_ff_siwis.wav`, `episode.wav`, `episode.json`) sont écrits dans `outputs/audio/kokoro/variante_28524/` pour ne pas écraser ceux de l'exemple 4 ; l'état du noyau est mémorisé en tête de cellule et restauré à la fin, car cette résolution redéfinit `compter_bascules`, `articles_corriges`, `manifeste` et `segments_episode`, que la synthèse de fin de carnet réutilise. Le reste est le code du groupe, tel quel.


In [23]:
# Exemple guide 4b (credite #28524) : phonemisation en contexte, accents et sigle
# Adaptation d'integration : cette resolution redefinit articles_corriges, manifeste, segments_episode...
# deja definis par l'exemple guide 4 et reutilises par la synthese de fin de carnet.
# L'etat du noyau est memorise ici et restaure en fin de cellule.
_etat_avant_4b = {k: v for k, v in globals().items() if not k.startswith("_")}
OUTPUT_DIR = OUTPUT_DIR / "variante_28524"   # adaptation : ne pas ecraser les fichiers de l'exemple guide 4 (episode.json, article_*.wav)
OUTPUT_DIR.mkdir(exist_ok=True)

import json

def compter_bascules(phonemes):
    """Nombre de passages vers l'anglais signales par espeak-ng dans une chaine de phonemes."""
    # Etape 2 : chaque bascule en plein mot est encadree par les marqueurs (en) ... (fr)
    return phonemes.count("(en)")

def texte_article(article):
    return article["title"] + ". " + article["summary"]

# Etape 1 : phonemes en-us et fr-fr pour chaque article de l'exemple guide 3
phonemes_origine = {}   # id -> phonemes fr-fr des textes SANS accents (reutilises a l'etape 3)
for article in test_articles:
    texte = texte_article(article)
    phonemes_en = kokoro.tokenizer.phonemize(texte, "en-us")
    phonemes_fr = kokoro.tokenizer.phonemize(texte, "fr-fr")
    phonemes_origine[article["id"]] = phonemes_fr
    print(f"Article {article['id']} : {texte}")
    print(f"  en-us : {phonemes_en}")
    print(f"  fr-fr : {phonemes_fr}   [{compter_bascules(phonemes_fr)} bascule(s) vers l'anglais]")

# Etape 3 : textes reecrits avec leurs accents
articles_accentues = [
    {"id": 1, "title": "IA et santé",
     "summary": "L'intelligence artificielle révolutionne le diagnostic médical."},
    {"id": 2, "title": "Voitures autonomes",
     "summary": "Les véhicules autonomes progressent rapidement sur les routes européennes."},
    {"id": 3, "title": "Quantique",
     "summary": "Les ordinateurs quantiques promettent des avancées majeures en cryptographie."},
]

print("\nEtape 3 : effet des accents (fr-fr, avant -> apres)")
for avant, apres in zip(test_articles, articles_accentues):
    ph_avant = phonemes_origine[avant["id"]]
    ph_apres = kokoro.tokenizer.phonemize(texte_article(apres), "fr-fr")
    print(f"Article {avant['id']} : bascules {compter_bascules(ph_avant)} -> {compter_bascules(ph_apres)}")
    mots_avant, mots_apres = ph_avant.split(), ph_apres.split()
    if len(mots_avant) != len(mots_apres):
        print("  (decoupage different, comparaison mot a mot impossible)")
        continue
    # Meme nombre de mots : on compare position par position
    mots_texte = texte_article(apres).split()
    for mot, pa, pb in zip(mots_texte, mots_avant, mots_apres):
        if pa != pb:
            type_changement = "bascule corrigee" if "(en)" in pa else "SILENCIEUX (aucun marqueur avant)"
            print(f"  {mot:<18} {pa:<28} -> {pb:<24} {type_changement}")

# Etape 4 : le sigle IA, lu comme un mot d'une syllabe, doit etre epele
print("\nEtape 4 : epellation du sigle")
for ecriture in ["IA et santé", "I A et santé", "I.A. et santé"]:
    print(f"  {ecriture:<16} -> {kokoro.tokenizer.phonemize(ecriture, 'fr-fr')}")
# "IA" donne "jˈa" (un mot). "I.A." donne "ˈi.ˈa." : deux lettres, chacune accentuee.
# "I A" donne "ˈi a" : la 2e lettre perd son accent, donc l'epellation est moins nette.

articles_corriges = [dict(article) for article in articles_accentues]
articles_corriges[0]["title"] = "I.A. et santé"
ph_ia = kokoro.tokenizer.phonemize(texte_article(articles_corriges[0]), "fr-fr")
assert "jˈa" not in ph_ia and "ˈi.ˈa." in ph_ia, "le sigle n'est pas epele"
print(f"  Verification : {ph_ia}")

# Etape 5 : synthese ff_siwis / fr-fr sur les textes corriges, puis manifeste episode.json
VOIX_FR, LANGUE_FR, PAUSE_ARTICLES = "ff_siwis", "fr-fr", 1.0

entrees, debut_episode = [], 0.0
segments_episode = []
for article in articles_corriges:
    texte = texte_article(article)
    samples, sr = kokoro.create(texte, voice=VOIX_FR, speed=1.0, lang=LANGUE_FR)
    nom_fichier = f"article_{article['id']}_{VOIX_FR}.wav"
    sf.write(str(OUTPUT_DIR / nom_fichier), samples, sr)
    duree = len(samples) / sr
    entrees.append({
        "id": article["id"],
        "fichier": nom_fichier,
        "voix": VOIX_FR,
        "langue": LANGUE_FR,
        "duree_s": round(duree, 3),
        # debut = somme des durees precedentes + pauses deja ecoulees
        "debut_s": round(debut_episode, 3),
    })
    print(f"Article {article['id']} : {duree:.2f}s, debut a {debut_episode:.2f}s")
    segments_episode += [samples, np.zeros(int(sr * PAUSE_ARTICLES), dtype=samples.dtype)]
    debut_episode += duree + PAUSE_ARTICLES

segments_episode = segments_episode[:-1]          # pas de silence apres le dernier article
episode = np.concatenate(segments_episode)
sf.write(str(OUTPUT_DIR / "episode.wav"), episode, sr)

manifeste = {
    "episode": "episode.wav",
    "pause_entre_articles_s": PAUSE_ARTICLES,
    "duree_totale_s": round(len(episode) / sr, 3),
    "articles": entrees,
}
with open(OUTPUT_DIR / "episode.json", "w", encoding="utf-8") as f:
    json.dump(manifeste, f, ensure_ascii=False, indent=2)

# Coherence : la fin du dernier article = duree totale de l'episode
derniere = entrees[-1]
assert abs((derniere["debut_s"] + derniere["duree_s"]) - manifeste["duree_totale_s"]) < 0.005
print(f"\nManifeste ecrit : episode.json ({len(entrees)} articles, {manifeste['duree_totale_s']:.2f}s)")

# Adaptation d'integration : restauration des noms partages avec l'exemple guide 4
for _nom, _valeur in _etat_avant_4b.items():
    globals()[_nom] = _valeur
del _etat_avant_4b, _nom, _valeur


Article 1 : IA et sante. L'intelligence artificielle revolutionne le diagnostic medical.
  en-us : ˌaɪˈeɪ ˈɛt sˈænt. ˈɛlˈɪntɪlˌɪdʒəns ˌɑːɹɾɪfˌɪsɪˈɛl ɹˌɛvəlˈuːʃən lə dˌaɪəɡnˈɑːstɪk mˈɛdɪkəl.
  fr-fr : jˈa e sˈɑ̃t. lɛ̃tɛliʒˈɑ̃s aʁtifisjˈɛl ʁəvolysjˈɔn lə djaɡnɔstˈik mədikˈal.   [0 bascule(s) vers l'anglais]
Article 2 : Voitures autonomes. Les vehicules autonomes progressent rapidement sur les routes europeennes.
  en-us : vˈɔɪtʃɚz ɔːtˈɑːnoʊmz. lˈɛs vˈɛhɪkjˌuːlz ɔːtˈɑːnoʊmz pɹəɡɹɪsˈɛnt ɹˈæpaɪdmənt sˈɜː lˈɛs ɹˈaʊts jˈʊɹɹəpˌiːnz.
  fr-fr : vwatˈyʁz otonˈom. le vəikˈylz otonˈom pʁɔɡʁˈɛs ʁapidmˈɑ̃ syʁ le ʁˈut (en)jˈʊəɹəpˌiːnz(fr).   [1 bascule(s) vers l'anglais]
Article 3 : Quantique. Les ordinateurs quantiques promettent des avancees majeures en cryptographie.
  en-us : kwɔntˈiːk. lˈɛs ˈɔːɹdᵻnɐtˈʊɹz kwɔntˈiːks pɹˈoʊmtənt dˈɛs ˈævɐnsˌiːz mɐdʒjˈuːɹəz ˈɛn kɹˈɪptəɡɹˌæfi.
  fr-fr : kɑ̃tˈik. lez ɔʁdinatˈœʁ kɑ̃tˈik pʁomˈɛt de (en)ˈavansˌiːz(fr) maʒˈœʁz ɑ̃ kʁiptɔɡʁafˈi.   [1 bascule(s) vers l'anglai

Article 1 : 4.93s, debut a 0.00s


Article 2 : 5.06s, debut a 5.93s


Article 3 : 4.67s, debut a 11.98s

Manifeste ecrit : episode.json (3 articles, 16.66s)


### Lecture du résultat — exemple guidé 4b

1. **Phonémisé en `en-us`, le français devient de l'anglais.** Les lignes `en-us` remplacent chaque mot par un mot anglais voisin (« IA » → ˌaɪˈeɪ, « routes » → ɹˈaʊts) : c'est le cas que les marqueurs `(en)` signalent en `fr-fr`.
2. **Compter les bascules ne suffit pas.** En `fr-fr` sans accents, deux mots seulement basculent vers l'anglais (« europeennes », « avancees »), et les accents les corrigent tous les deux (bascules 1 → 0 pour les articles 2 et 3). Mais six autres mots changent de prononciation sans qu'aucun marqueur ne l'ait signalé. Quatre d'entre eux étaient mal lus sans leur accent : « sante » (sˈɑ̃t, sans le é final), « revolutionne », « medical » et « vehicules » (ə au lieu de e). L'article 1 reste à 0 bascule avant et après, alors qu'il contenait trois mots mal prononcés.
3. **Deux changements viennent du mot suivant.** « routes » (ʁˈut → ʁˈutz) et « des » (de → dez) ne portent aucun accent. Ils changent parce que le mot qui les suit, une fois corrigé, est prononcé en français et commence par une voyelle (øʁopeˈɛn, avɑ̃sˈe), ce qui déclenche la liaison, obligatoire après « des », facultative après « routes ». Une comparaison mot à mot, comme celle de l'exemple 4, ne peut pas voir ces changements : c'est l'apport de la phonémisation en contexte.
4. **Le sigle.** « IA » est lu comme un mot (jˈa) ; « I A » et « I.A. » font épeler les deux lettres (ˈi a, ˈi.ˈa.). Le groupe retient la forme pointée « I.A. », et la vérification sur l'article entier confirme l'épellation.
5. **L'épisode.** Trois articles de 4,93 s, 5,06 s et 4,67 s séparés par 1 s de silence, soit 16,66 s ; les débuts du manifeste (0,00 s, 5,93 s, 11,98 s) sont, à l'arrondi près, la somme des durées et des pauses précédentes.

Limites : trois articles, une voix (`ff_siwis`), un seul phonémiseur (espeak-ng). La prononciation « corrigée » est celle d'espeak-ng : l'écoute des fichiers de `variante_28524/` reste le juge.

### Exercice 2bis - Homographes : la phonémisation lit-elle la grammaire ?

L'exemple guidé 4b montre que les phonèmes d'un mot changent **en contexte** (liaisons). Le contexte décide aussi des **homographes** : dans « les poules du couvent couvent », le même mot écrit se prononce /kuvɑ̃/ puis /kuv/. Lire mot à mot ne suffit pas ; il faut savoir que le second est un verbe. Quels homographes la phonémisation `fr-fr` de Kokoro (espeak-ng) départage-t-elle ?

**Étapes** :
1. Écrire `phonemes_mots(phrase)` : phonémiser la phrase entière avec `kokoro.tokenizer.phonemize(phrase, "fr-fr")`, retirer les marques d'accentuation `ˈ` et `ˌ`, retirer la ponctuation, puis découper aux espaces
2. Pour chaque phrase de `HOMOGRAPHES`, vérifier que les prononciations attendues apparaissent **dans l'ordre** parmi les mots phonémisés — remplir `verdicts` (dict phrase → `True`/`False`) et `score` (nombre de phrases bien lues)
3. Regrouper les échecs : quelle sorte d'homographe la phonémisation ne sait-elle pas départager ? Relever aussi, avec `compter_bascules`, les phrases dont les phonèmes contiennent une bascule `(en)` : un mot français peut y être lu comme de l'anglais — remplir `phrases_avec_bascule` (liste de phrases)

**Indices** :
- `# Indice` : les prononciations attendues sont notées avec l'inventaire d'espeak-ng (par exemple `o` dans « violent »), pour que la comparaison porte sur le choix de prononciation et non sur la notation
- `# Indice` : « dans l'ordre » : cherchez la première prononciation attendue, puis la suivante **parmi les mots qui suivent**


In [24]:
# Exercice 2bis : homographes -- la phonemisation lit-elle la grammaire ?
# TODO etudiant : phonemiser chaque phrase et verifier les prononciations attendues, dans l'ordre
# Etape 1 : phonemes_mots(phrase) -> liste de mots phonemises, sans accents ni ponctuation
# Etape 2 : prononciations attendues trouvees dans l'ordre -> verdicts (dict phrase -> bool), score
# Etape 3 : regrouper les echecs ; compter_bascules() -> phrases_avec_bascule (liste)
# Indice : les formes attendues suivent la notation d'espeak-ng ; seul le choix de prononciation compte
HOMOGRAPHES = {
    "Les poules du couvent couvent leurs oeufs.": ["kuvɑ̃", "kuv"],
    "Le fils de Marie coud avec des fils de soie.": ["fis", "fil"],
    "Le vent souffle de l'est. Il est tard.": ["lɛst", "ɛ"],
    "Le président parle. Ils président la séance.": ["pʁezidɑ̃", "pʁezid"],
    "Nous portions les portions de riz.": ["pɔʁtjɔ̃", "pɔʁsjɔ̃"],
    "Les violents violent la loi.": ["vjolɑ̃", "vjol"],
    "Ils négligent un détail négligent.": ["neɡliʒ", "neɡliʒɑ̃"],
    "Il est content. Ils content une histoire.": ["kɔ̃tɑ̃", "kɔ̃t"],
    "Six enfants. Six garçons. J'en ai six.": ["siz", "si", "sis"],
    "Tous les jours, ils viennent tous.": ["tu", "tus"],
}
verdicts = None  # TODO etudiant
score = None  # TODO etudiant
phrases_avec_bascule = None  # TODO etudiant
print("Exercice 2bis a completer : homographes")


Exercice 2bis a completer : homographes


## Section 5 : Mode interactif

Cette section offre une interface interactive pour experimenter avec Kokoro TTS en temps reel.

### Fonctionnalite

En mode interactif, vous pouvez :
- Saisir n'importe quel texte a synthetiser
- Choisir parmi les voix disponibles
- Obtenir immediatement l'audio genere
- Sauvegarder les résultats avec horodatage

### Contraintes d'exécution

| Mode | Comportement |
|------|---------------|
| **Interactive** | Invite de commande `input()` pour saisie utilisateur |
| **Batch** | Desactive automatiquement (`skip_widgets=True`) |
| **MCP/Papermill** | Desactive (stdin non disponible) |

> **Note** : En mode batch, cette section est automatiquement sautee pour ne pas bloquer l'exécution.

In [25]:
# Mode interactif
if notebook_mode == "interactive" and not skip_widgets:
    print("MODE INTERACTIF")
    print("=" * 50)
    print("\nEntrez un texte a synthetiser avec Kokoro :")
    print("(Laissez vide pour passer a la suite)")

    try:
        user_text = input("\nVotre texte : ")

        if user_text.strip() and kokoro_loaded:
            user_voice = input(f"Voix [{voice}] : ").strip() or voice

            print(f"\nGeneration avec Kokoro ({user_voice})...")
            start_time = time.time()

            if use_onnx:
                samples, sr_out = kokoro.create(user_text, voice=user_voice, speed=1.0)
            else:
                generator = pipeline(user_text, voice=user_voice, speed=1.0)
                all_samples = []
                sr_out = 24000
                for gs, ps, audio_chunk in generator:
                    all_samples.append(audio_chunk)
                samples = np.concatenate(all_samples) if all_samples else np.array([])

            gen_time = time.time() - start_time
            print(f"Duree : {len(samples)/sr_out:.1f}s | Temps : {gen_time:.2f}s")
            display_audio_array(samples, sr_out)

            if save_results:
                ts = datetime.now().strftime('%Y%m%d_%H%M%S')
                filepath = OUTPUT_DIR / f"custom_{user_voice}_{ts}.wav"
                sf.write(str(filepath), samples, sr_out)
                print(f"Sauvegarde : {filepath.name}")
        else:
            print("Mode interactif ignore")

    except (KeyboardInterrupt, EOFError):
        print("Mode interactif interrompu")
    except Exception as e:
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("Mode interactif non disponible (execution automatisee)")
        else:
            print(f"Erreur : {error_type} - {str(e)[:100]}")
else:
    print("Mode batch - Interface interactive desactivee")


Mode batch - Interface interactive desactivee


## Bonnes pratiques et guide de decision

### Arbre de decision : local vs cloud

| Question | Oui -> | Non -> |
|----------|--------|--------|
| Données sensibles ? | Local (Kokoro) | Continuer |
| Budget limite ? | Local (Kokoro) | Continuer |
| Qualite maximale requise ? | Cloud (OpenAI HD) | Continuer |
| GPU disponible ? | Local (Kokoro) | Cloud (OpenAI tts-1) |
| Volume > 1M chars/mois ? | Local (Kokoro) | Cloud (OpenAI tts-1) |

### Optimisation Kokoro

| Technique | Effet mesuré dans ce carnet | Description |
|-----------|--------|-------------|
| Backend ONNX | Non mesuré ici | Inference optimisee sans PyTorch |
| Batch generation | Non mesuré ici | Regrouper les textes courts |
| GPU | Non mesuré ici (pas de comparaison CPU / GPU dans ce carnet) | Recommande pour la production |

In [26]:
# Statistiques de session et prochaines etapes
print("STATISTIQUES DE SESSION")
print("=" * 45)

print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Modele : {kokoro_model}")
print(f"Backend : {'ONNX' if use_onnx else 'PyTorch'}")
print(f"Voix par defaut : {voice}")
print(f"Modele charge : {'Oui' if kokoro_loaded else 'Non'}")

if gpu_available:
    vram_current = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"VRAM utilisee : {vram_current:.2f} GB")

if save_results:
    saved = list(OUTPUT_DIR.glob('*'))
    total_size = sum(f.stat().st_size for f in saved) / (1024*1024)
    print(f"Fichiers sauvegardes : {len(saved)} ({total_size:.1f} MB) dans {OUTPUT_DIR.relative_to(GENAI_ROOT).as_posix()}")

_whisper_model = None  # libère Whisper (exercices 1 et 2 terminés)

# Liberation memoire
if kokoro_loaded:
    print(f"\nLiberation du modele...")
    if use_onnx:
        del kokoro
    else:
        del pipeline
    gc.collect()
    if gpu_available:
        torch.cuda.empty_cache()
    print(f"Memoire liberee")

print(f"\nPROCHAINES ETAPES")
print(f"1. Explorer le TTS avance avec Chatterbox (02-1)")
print(f"2. Decouvrir le voice cloning avec XTTS (02-2)")
print(f"3. Comparer tous les modeles TTS et STT (03-1)")
print(f"4. Construire un pipeline vocal complet (03-2)")

print(f"\nNotebook Kokoro TTS termine - {datetime.now().strftime('%H:%M:%S')}")


STATISTIQUES DE SESSION
Date : 2026-11-28 03:52:49
Modele : hexgrad/Kokoro-82M
Backend : ONNX
Voix par defaut : af_heart
Modele charge : Oui
Fichiers sauvegardes : 52 (14.6 MB) dans outputs/audio/kokoro

Liberation du modele...


Memoire liberee

PROCHAINES ETAPES
1. Explorer le TTS avance avec Chatterbox (02-1)
2. Decouvrir le voice cloning avec XTTS (02-2)
3. Comparer tous les modeles TTS et STT (03-1)
4. Construire un pipeline vocal complet (03-2)

Notebook Kokoro TTS termine - 03:52:49


### Statistiques
Les mesures de cette exécution sont affichées ci-dessus. Les performances dépendent du matériel, du backend et du texte. Le modèle sera rechargé pour le dialogue guidé.

### Exemple guidé 5 : Dialogue synthétique à deux voix

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20681), intégrée comme exemple guidé.*

Les voix testées en section 3 servent ici à construire un dialogue : deux interlocuteurs, une voix chacun, une pause entre deux répliques. La même technique sert aux podcasts générés, aux livres audio à plusieurs personnages et aux assistants vocaux à personnalités distinctes.

**Objectif** : assembler un seul fichier audio à partir de répliques synthétisées séparément.

**Étapes** :
1. Écrire le dialogue comme une liste de couples `(interlocuteur, réplique)`, en alternance A-B
2. Associer une voix à chaque interlocuteur : `af_heart` pour A, `am_adam` pour B
3. Synthétiser chaque réplique, puis insérer 500 ms de silence (`np.zeros(int(sample_rate * 0.5))`) entre deux répliques
4. Concaténer les segments avec `np.concatenate`, sauvegarder `synthetic_dialogue.wav` et afficher les statistiques

**Points clés du code** :
- la cellule de statistiques plus haut libère le modèle (`del kokoro`) : l'exemple le recharge depuis le cache local avant de synthétiser
- toutes les voix Kokoro produisent du 24 000 Hz, ce qui permet de concaténer les segments sans rééchantillonnage
- la liste `audio_segments` alterne répliques et silences : sa structure garde la trace de qui parle à quel moment, ce que l'exercice 3 exploite

In [27]:
import numpy as np
import soundfile as sf
from kokoro_onnx import Kokoro

# Le modele a ete libere dans la cellule de statistiques : on le recharge
cache_dir = Path.cwd() / '.cache' / 'kokoro-onnx'
kokoro = Kokoro(str(cache_dir / 'kokoro-v1.0.onnx'), str(cache_dir / 'voices-v1.0.bin'))

# Dialogue (alternance A-B)
dialogue = [
    ("A", "Hello, welcome to this synthetic dialogue."),
    ("B", "Thank you, it's great to be here."),
    ("A", "Let's explore text-to-speech synthesis together."),
    ("B", "I agree, this technology is fascinating."),
    ("A", "Do you think it will replace human voices?"),
    ("B", "Not completely, but it will be very useful."),
]

# Une voix Kokoro par speaker
voices = {
    "A": "af_heart",
    "B": "am_adam"
}

audio_segments = []
pause_duration = 0.5  # 500ms de pause entre les interventions

for i, (speaker, text) in enumerate(dialogue):
    print(f"Generating {speaker}: {text[:30]}...")

    # Generer l'audio avec la voix du speaker
    samples, sample_rate = kokoro.create(text, voice=voices[speaker], speed=1.0)

    audio_segments.append(samples)

    # Pause apres chaque intervention sauf la derniere
    if i < len(dialogue) - 1:
        silence = np.zeros(int(sample_rate * pause_duration))
        audio_segments.append(silence)

# Concatener tous les segments
full_dialogue_audio = np.concatenate(audio_segments)

if full_dialogue_audio is None:
    print("Dialogue non genere (exercice a completer ou service indisponible)")
    print(f"Nombre de tours definis: {len(dialogue)}")
else:
    total_duration = len(full_dialogue_audio) / sample_rate
    num_turns = len(dialogue)
    print(f"\nDialogue Statistics:")
    print(f"  Total duration: {total_duration:.1f}s")
    print(f"  Number of turns: {num_turns}")
    print(f"  Average turn length: {total_duration/num_turns:.1f}s")

    # Sauvegarder le resultat
    output_path = OUTPUT_DIR / "synthetic_dialogue.wav"
    sf.write(str(output_path), full_dialogue_audio, sample_rate)

    print(f"\nSaved: {output_path.name}")

    # Ecouter le resultat
    from IPython.display import Audio
    display_audio_array(full_dialogue_audio, sample_rate)

Generating A: Hello, welcome to this synthet...


Generating B: Thank you, it's great to be he...


Generating A: Let's explore text-to-speech s...


Generating B: I agree, this technology is fa...


Generating A: Do you think it will replace h...


Generating B: Not completely, but it will be...



Dialogue Statistics:
  Total duration: 15.9s
  Number of turns: 6
  Average turn length: 2.6s

Saved: synthetic_dialogue.wav


<audio embedded (audio/wav, 763116 bytes)>

### Exemple guidé 6 (crédité #21036) : Sous-titres exacts et troisième voix

Pour produire des sous-titres, le carnet 01-2 devait **estimer** les instants de parole à partir d'une transcription Whisper. Ici, le dialogue est synthétisé réplique par réplique : la durée de chaque segment est connue **exactement**, à l'échantillon près. Les sous-titres se calculent, ils ne s'estiment pas.

**Étapes** :
1. À partir de `dialogue` et `audio_segments` (exemple guidé 5), calculer l'instant de début et de fin de chaque réplique en secondes, en cumulant les longueurs des segments, silences compris
2. Écrire `dialogue.srt` : une entrée par réplique, préfixée de l'interlocuteur (`A :`, `B :`)
3. Vérifier la cohérence : la fin de la dernière réplique doit coïncider avec la durée totale de `full_dialogue_audio`
4. Ajouter un troisième interlocuteur C avec une voix britannique (`bf_emma` ou `bm_george`) et deux répliques, puis régénérer le dialogue et ses sous-titres
5. Faire varier `speed` selon l'intention de la réplique (une question un peu plus lente, une exclamation un peu plus rapide) et vérifier que les sous-titres restent alignés

**Indices** :
- `# Indice` : un instant en secondes vaut `nombre_d_echantillons / sample_rate` ; dans `audio_segments`, les répliques occupent les positions paires et les silences les positions impaires
- `# Indice` : le format SRT d'un instant est `HH:MM:SS,mmm` ; calculez-le en millisecondes entières avec des `divmod` successifs
- `# Indice` : `speed` reste entre 0.5 et 2.0 (borne de `kokoro.create()`)

> **Exemple guidé (crédité #21036).** La résolution ci-dessous est celle du groupe de la PR #21036, conservée telle quelle : horaires exacts par cumul des échantillons (répliques aux positions paires, silences aux impaires), écriture du `.srt`, contrôle de cohérence sur la durée totale, puis dialogue à trois voix avec vitesses d'intention. Le nouveau **Exercice 3** qui suit reprend ce SRT pour lui imposer une contrainte de **largeur de ligne** — le format lisible, pas seulement le format juste.


In [28]:
# Exemple guidé 6 : version initiale puis ajout de deux répliques britanniques
def instant_srt(secondes):
    millis = round(secondes * 1000)
    heures, reste = divmod(millis, 3600000)
    minutes, reste = divmod(reste, 60000)
    secondes, millis = divmod(reste, 1000)
    return f'{heures:02d}:{minutes:02d}:{secondes:02d},{millis:03d}'

def calculer_horaires(dialogue, audio_segments, sample_rate):
    assert sample_rate > 0 and len(audio_segments) == 2 * len(dialogue) - 1
    position, horaires = 0, []
    for i, segment in enumerate(audio_segments):
        fin = position + len(segment)
        if i % 2 == 0:
            speaker, texte = dialogue[i // 2]
            horaires.append((speaker, texte, position/sample_rate, fin/sample_rate))
        position = fin
    return horaires

def sauvegarder_dialogue(nom, dialogue, segments, sr):
    horaires = calculer_horaires(dialogue, segments, sr)
    audio = np.concatenate(segments)
    assert abs(horaires[-1][3] - len(audio)/sr) <= 1/sr
    assert abs(round(horaires[-1][3]*1000)/1000 - len(audio)/sr) <= 0.0005 + 1e-12
    blocs = [f'{i}\n{instant_srt(debut)} --> {instant_srt(fin)}\n{speaker} : {texte}\n'
             for i, (speaker, texte, debut, fin) in enumerate(horaires, 1)]
    (OUTPUT_DIR / f'{nom}.srt').write_text('\n'.join(blocs), encoding='utf-8')
    sf.write(str(OUTPUT_DIR / f'{nom}.wav'), audio, sr)
    print(nom, ':', len(horaires), 'répliques ; durée =', len(audio)/sr, 's ; alignement vérifié')
    print('\n'.join(blocs))
    display_audio_array(audio, sr)
    return horaires

horaires_initial = sauvegarder_dialogue('dialogue', dialogue, audio_segments, sample_rate)
dialogue_3_voix = dialogue[:2] + [('C', 'May I join your discussion?')] + dialogue[2:] + [('C', 'Brilliant! This is a useful demonstration!')]
voices_3_voix = {**voices, 'C': 'bf_emma'}
segments_3, sr_3 = [], None
for i, (speaker, texte) in enumerate(dialogue_3_voix):
    vitesse = 0.9 if '?' in texte else 1.1 if '!' in texte else 1.0
    langue = 'en-gb' if speaker == 'C' else 'en-us'
    audio, sr = kokoro.create(texte, voice=voices_3_voix[speaker], speed=vitesse, lang=langue)
    if sr_3 is None: sr_3 = sr
    assert sr == sr_3
    print(speaker, 'speed =', vitesse, 'lang =', langue)
    segments_3.append(audio)
    if i < len(dialogue_3_voix)-1:
        segments_3.append(np.zeros(round(sr*0.5), dtype=np.float32))
horaires_3 = sauvegarder_dialogue('dialogue_3_voix', dialogue_3_voix, segments_3, sr_3)

dialogue : 6 répliques ; durée = 15.897333333333334 s ; alignement vérifié
1
00:00:00,000 --> 00:00:02,560
A : Hello, welcome to this synthetic dialogue.

2
00:00:03,060 --> 00:00:04,681
B : Thank you, it's great to be here.

3
00:00:05,181 --> 00:00:07,677
A : Let's explore text-to-speech synthesis together.

4
00:00:08,177 --> 00:00:10,524
B : I agree, this technology is fascinating.

5
00:00:11,024 --> 00:00:13,136
A : Do you think it will replace human voices?

6
00:00:13,636 --> 00:00:15,897
B : Not completely, but it will be very useful.



<audio embedded (audio/wav, 763116 bytes)>

A speed = 1.0 lang = en-us


B speed = 1.0 lang = en-us


C speed = 0.9 lang = en-gb


A speed = 1.0 lang = en-us


B speed = 1.0 lang = en-us


A speed = 0.9 lang = en-us


B speed = 1.0 lang = en-us


C speed = 1.1 lang = en-gb
dialogue_3_voix : 8 répliques ; durée = 21.228 s ; alignement vérifié
1
00:00:00,000 --> 00:00:02,560
A : Hello, welcome to this synthetic dialogue.

2
00:00:03,060 --> 00:00:04,681
B : Thank you, it's great to be here.

3
00:00:05,181 --> 00:00:07,123
C : May I join your discussion?

4
00:00:07,623 --> 00:00:10,119
A : Let's explore text-to-speech synthesis together.

5
00:00:10,619 --> 00:00:12,965
B : I agree, this technology is fascinating.

6
00:00:13,465 --> 00:00:15,705
A : Do you think it will replace human voices?

7
00:00:16,205 --> 00:00:18,467
B : Not completely, but it will be very useful.

8
00:00:18,967 --> 00:00:21,228
C : Brilliant! This is a useful demonstration!



<audio embedded (audio/wav, 1018988 bytes)>

### Analyse de l’exemple guidé 6

Le dialogue initial contient **6 répliques** et dure **15,897333 s**. La version à trois voix contient **8 répliques**, dont deux pour C avec `bf_emma` et la langue `en-gb`, et dure **21,228 s**. Les pauses intermédiaires durent 0,5 s ; aucune pause n’est ajoutée après la dernière réplique.

Les questions utilisent `speed=0.9`, les exclamations `speed=1.1` et les autres répliques `speed=1.0`. Les horaires sont recalculés à partir des segments effectivement générés : la variation de vitesse ne repose donc pas sur une estimation de leur durée.

Les fins SRT sont 15,897 s et 21,228 s. L’écart avec les WAV est respectivement d’environ **0,333 ms et 0 ms**, compatible avec l’arrondi du format SRT. Les horaires décrivent les limites des segments synthétisés, qui peuvent inclure un peu de silence interne ; ils ne constituent pas un alignement mot à mot.


### Exercice 3 : Sous-titres à largeur bornée — le SRT lisible

L'exemple guidé 6 produit un SRT **juste** : les horaires collent à l'échantillon près. Un sous-titre diffusé doit aussi être **lisible** : les praticiens bornent la ligne à ~42 caractères (la largeur d'un écran de télévision analogique, restée la norme). Découper une réplique en lignes courtes sans couper un mot, sans dépasser la borne — et sans décaler les horaires.

**Étapes** :
1. Écrire `decouper(texte, borne=42)` : découpe aux espaces (jamais au milieu d'un mot), remplit chaque ligne au maximum sans dépasser `borne` — gérer le cas d'un mot unique plus long que la borne (ligne à lui seul)
2. Réécrire chaque entrée du dialogue de l'exemple guidé 6 avec son préfixe interlocuteur et ses lignes découpées — remplir `entrees_reformatees` (liste de listes de lignes, une sous-liste par entrée SRT)
3. Vérifier : `largeur_maximale` (le plus long des lignes produites, hors entrées à mot unique plus long que la borne) et `lignes_par_entree` (liste d'entiers) — puis vérifier que les horaires de `dialogue.srt` de l'exemple guidé 6 n'ont pas bougé d'un échantillon

**Indices** :
- `# Indice` : préfixe `A : ` et ligne de texte partagent la même ligne SRT — la borne s'applique au total
- `# Indice` : découpez glouton (mot suivant tant qu'il tient) ; un `"".join` de contrôle par ligne évite de recompter deux fois

In [29]:
# Exercice 3 : sous-titres a largeur bornee -- le SRT lisible
# TODO etudiant : decouper les entrees du SRT en lignes <= 42 caracteres sans couper un mot
# Etape 1 : decouper(texte, borne=42) glouton aux espaces -> lignes
# Etape 2 : entrees reformatees avec prefixe interlocuteur -> entrees_reformatees (liste de listes)
# Etape 3 : plus longue ligne -> largeur_maximale ; lignes par entree -> lignes_par_entree
# Indice : le prefixe "A : " compte dans la borne ; un mot seul plus long que la borne fait ligne a part
entrees_reformatees = None  # TODO etudiant
largeur_maximale = None  # TODO etudiant
lignes_par_entree = None  # TODO etudiant
print("Exercice 3 a completer : SRT a largeur bornee")

Exercice 3 a completer : SRT a largeur bornee


### Exemple guidé 6b (crédité #28524) : la même consigne, vérifiée à l'échantillon

La consigne est celle de l'exemple guidé 6 ; la solution ci-dessous vient d'un **second groupe**, celui de la PR #28524, conservée quasi verbatim. Elle diffère de l'exemple 6 sur deux points. La vitesse d'intention dépend de la **ponctuation finale** de la réplique (`?` en fin de réplique ralentit à 0,9, `!` accélère à 1,1), là où l'exemple 6 cherche `?` ou `!` n'importe où dans la réplique. Et la vérification ne s'arrête pas à la durée totale : chaque réplique est retrouvée **à l'échantillon près** dans l'audio concaténé (`np.array_equal`), et chaque pause est contrôlée silencieuse.

Adaptations d'intégration, déclarées dans la PR : titre et message de la branche vide reformulés ; les fichiers (`dialogue.srt`, `dialogue_3voix.srt`, `dialogue_3voix.wav`) sont écrits dans `outputs/audio/kokoro/variante_28524/` pour ne pas écraser ceux de l'exemple 6 ; l'état du noyau est mémorisé en tête de cellule et restauré à la fin, car cette résolution redéfinit `instant_srt`, `calculer_horaires` et `horaires_3`, que la synthèse de fin de carnet et l'exercice 3bis réutilisent. Le reste est le code du groupe, tel quel.


In [30]:
# Exemple guide 6b (credite #28524) : sous-titres exacts, verifies a l'echantillon
# Adaptation d'integration : cette resolution redefinit instant_srt, calculer_horaires, segments_3, sr_3...
# deja definis par l'exemple guide 6 et reutilises par la synthese de fin de carnet.
# L'etat du noyau est memorise ici et restaure en fin de cellule.
_etat_avant_6b = {k: v for k, v in globals().items() if not k.startswith("_")}
OUTPUT_DIR = OUTPUT_DIR / "variante_28524"   # adaptation : ne pas ecraser les fichiers de l'exemple guide 6 (dialogue.srt)
OUTPUT_DIR.mkdir(exist_ok=True)

def instant_srt(secondes):
    """Formate un instant en HH:MM:SS,mmm."""
    # Calcul en millisecondes entieres pour eviter les erreurs d'arrondi des flottants
    millisecondes = round(secondes * 1000)
    secondes_entieres, ms = divmod(millisecondes, 1000)
    minutes, s = divmod(secondes_entieres, 60)
    h, m = divmod(minutes, 60)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def calculer_horaires(dialogue, audio_segments, sample_rate):
    """Liste de (interlocuteur, replique, debut_s, fin_s) pour chaque replique."""
    # Etape 1 : on cumule les longueurs en echantillons (exact), on convertit a la fin.
    # audio_segments alterne replique (indice pair) et silence (indice impair).
    horaires, position = [], 0
    for i, segment in enumerate(audio_segments):
        if i % 2 == 0:
            interlocuteur, replique = dialogue[i // 2]
            horaires.append((interlocuteur, replique,
                             position / sample_rate,
                             (position + len(segment)) / sample_rate))
        position += len(segment)          # les silences avancent la position sans produire d'entree
    return horaires

def ecrire_srt(horaires, chemin):
    """Ecrit un fichier SRT : une entree par replique, prefixee de l'interlocuteur."""
    blocs = []
    for numero, (interlocuteur, replique, debut, fin) in enumerate(horaires, start=1):
        blocs.append(f"{numero}\n{instant_srt(debut)} --> {instant_srt(fin)}\n{interlocuteur} : {replique}\n")
    chemin.write_text("\n".join(blocs), encoding="utf-8")

horaires = calculer_horaires(dialogue, audio_segments, sample_rate)

if horaires:
    # Etapes 2 et 3 : ecrire dialogue.srt et verifier la fin de la derniere replique
    ecrire_srt(horaires, OUTPUT_DIR / "dialogue.srt")
    duree_totale = len(full_dialogue_audio) / sample_rate
    fin_derniere = horaires[-1][3]
    assert abs(fin_derniere - duree_totale) < 1 / sample_rate, "sous-titres decales"
    print(f"dialogue.srt ecrit ({len(horaires)} repliques) - fin {fin_derniere:.3f}s = duree totale {duree_totale:.3f}s")
    print((OUTPUT_DIR / "dialogue.srt").read_text(encoding="utf-8").split("\n\n")[0])

    # Etape 4 : troisieme interlocuteur C (voix britannique) et deux repliques
    dialogue_3 = [
        ("A", "Hello, welcome to this synthetic dialogue."),
        ("B", "Thank you, it's great to be here."),
        ("C", "Good evening, may I join the conversation?"),
        ("A", "Of course, we were just exploring text-to-speech synthesis."),
        ("B", "I agree, this technology is fascinating!"),
        ("C", "Splendid! Does it really sound natural?"),
        ("A", "Do you think it will replace human voices?"),
        ("B", "Not completely, but it will be very useful."),
    ]
    voices_3 = {"A": "af_heart", "B": "am_adam", "C": "bf_emma"}

    # Etape 5 : speed selon l'intention de la replique (bornes de kokoro.create : 0.5 a 2.0)
    def vitesse_replique(replique):
        if replique.rstrip().endswith("?"):
            return 0.9      # question : un peu plus lente
        if replique.rstrip().endswith("!"):
            return 1.1      # exclamation : un peu plus rapide
        return 1.0

    def synthetiser_dialogue(dialogue, voices, pause=0.5):
        """Renvoie (audio complet, segments alternes replique/silence, sample_rate, vitesses)."""
        segments, vitesses = [], []
        for i, (interlocuteur, replique) in enumerate(dialogue):
            vitesse = vitesse_replique(replique)
            samples, sr = kokoro.create(replique, voice=voices[interlocuteur], speed=vitesse)
            segments.append(samples)
            vitesses.append(vitesse)
            if i < len(dialogue) - 1:
                segments.append(np.zeros(int(sr * pause), dtype=samples.dtype))
        return np.concatenate(segments), segments, sr, vitesses

    audio_3, segments_3, sr_3, vitesses_3 = synthetiser_dialogue(dialogue_3, voices_3)
    horaires_3 = calculer_horaires(dialogue_3, segments_3, sr_3)
    ecrire_srt(horaires_3, OUTPUT_DIR / "dialogue_3voix.srt")
    sf.write(str(OUTPUT_DIR / "dialogue_3voix.wav"), audio_3, sr_3)

    # Verification d'alignement : le morceau d'audio [debut, fin] doit etre exactement la
    # replique synthetisee, et ce qui separe deux repliques doit etre du silence
    fin_precedente = 0
    for (interlocuteur, replique, debut, fin), segment in zip(horaires_3, segments_3[::2]):
        i_debut, i_fin = round(debut * sr_3), round(fin * sr_3)
        assert np.array_equal(audio_3[i_debut:i_fin], segment), f"decalage sur : {replique}"
        assert not audio_3[fin_precedente:i_debut].any(), "du son dans une pause"
        fin_precedente = i_fin
    assert abs(horaires_3[-1][3] - len(audio_3) / sr_3) < 1 / sr_3

    print(f"\nDialogue a 3 voix : {len(audio_3) / sr_3:.1f}s, sous-titres alignes (verifie a l'echantillon)")
    print(f"{'Qui':<4}{'Debut':<14}{'Fin':<14}{'Speed':<7}Replique")
    for (qui, replique, debut, fin), vitesse in zip(horaires_3, vitesses_3):
        print(f"{qui:<4}{instant_srt(debut):<14}{instant_srt(fin):<14}{vitesse:<7}{replique}")
    display_audio_array(audio_3, sr_3)
else:
    print("Aucun horaire calcule : calculer_horaires() a renvoye une liste vide")
    print(f"Repliques : {len(dialogue)}, segments (repliques et silences) : {len(audio_segments)}")

# Adaptation d'integration : restauration des noms partages avec l'exemple guide 6
for _nom, _valeur in _etat_avant_6b.items():
    globals()[_nom] = _valeur
del _etat_avant_6b, _nom, _valeur


dialogue.srt ecrit (6 repliques) - fin 15.897s = duree totale 15.897s
1
00:00:00,000 --> 00:00:02,560
A : Hello, welcome to this synthetic dialogue.



Dialogue a 3 voix : 22.3s, sous-titres alignes (verifie a l'echantillon)
Qui Debut         Fin           Speed  Replique
A   00:00:00,000  00:00:02,560  1.0    Hello, welcome to this synthetic dialogue.
B   00:00:03,060  00:00:04,681  1.0    Thank you, it's great to be here.
C   00:00:05,181  00:00:07,677  0.9    Good evening, may I join the conversation?
A   00:00:08,177  00:00:11,399  1.0    Of course, we were just exploring text-to-speech synthesis.
B   00:00:11,899  00:00:14,032  1.1    I agree, this technology is fascinating!
C   00:00:14,532  00:00:16,793  0.9    Splendid! Does it really sound natural?
A   00:00:17,293  00:00:19,533  0.9    Do you think it will replace human voices?
B   00:00:20,033  00:00:22,295  1.0    Not completely, but it will be very useful.


<audio embedded (audio/wav, 1070188 bytes)>

### Lecture du résultat — exemple guidé 6b

1. **Dialogue initial.** 6 répliques, dernier sous-titre terminé à 15,897 s, exactement la durée de l'audio, comme dans l'exemple 6.
2. **La vitesse d'intention suit la ponctuation finale.** Trois répliques finissent par « ? » et sont ralenties à 0,9 ; une finit par « ! » et est accélérée à 1,1. « Splendid! Does it really sound natural? » contient les deux signes : seule la fin compte, elle est donc traitée comme une question (0,9).
3. **L'alignement est vérifié à l'échantillon.** Les 8 répliques se retrouvent à l'identique dans l'audio concaténé, et les 7 pauses de 0,5 s sont silencieuses : les horaires du tableau (22,3 s au total) sont exacts réplique par réplique, pas seulement cohérents en durée totale.
4. **La voix britannique est phonémisée en anglais américain.** `kokoro.create` est appelé sans `lang`, dont la valeur par défaut dans kokoro-onnx 0.6.1 est `en-us`, y compris pour `bf_emma`. L'exemple 6 passe `en-gb` pour C. Ici, le timbre de C est britannique mais sa phonémisation est américaine ; l'écoute dira si l'écart s'entend.
5. **Pour l'exercice 3bis.** L'état restauré en fin de cellule rend à `horaires_3` les horaires de l'exemple 6 (8 répliques, 21,228 s) : c'est sur eux que porte l'exercice.

Limites : une seule synthèse par réplique ; l'intention d'une réplique ne se réduit pas à sa ponctuation finale, et deux vitesses d'intention seulement sont testées.

### Exercice 3bis - Vitesse de lecture : le sous-titre laisse-t-il le temps de lire ?

Les exemples guidés 6 et 6b produisent des sous-titres **justes** : chaque entrée s'affiche exactement pendant sa réplique. L'exercice 3 les rend **lisibles en largeur**. Reste le temps de lecture : les chartes de sous-titrage bornent la vitesse de lecture, souvent entre 15 et 20 caractères par seconde selon la langue et le public. On retient ici 17 caractères par seconde, espaces et préfixe d'interlocuteur compris. Une parole de synthèse à la vitesse 1.0 tient-elle cette borne ?

**Étapes** :
1. Pour chaque entrée de `horaires_3` (le dialogue à trois voix de l'exemple guidé 6), calculer la vitesse de lecture `len(f"{interlocuteur} : {replique}") / (fin - debut)` — remplir `cps_initial` (liste, dans l'ordre du dialogue) et `trop_rapides` (nombre d'entrées au-dessus de `BORNE_LECTURE`)
2. Prolonger chaque entrée trop rapide dans le silence qui la suit, sans jamais avancer son début : elle finit au plus tard 0,1 s avant le début de l'entrée suivante (la dernière, au plus tard 0,5 s après la fin de l'audio) — remplir `horaires_prolonges` (même forme que `horaires_3`) et `encore_trop_rapides` (nombre d'entrées toujours au-dessus de la borne)
3. Pour les entrées qui résistent, c'est la parole elle-même qui va trop vite. Avec la loi de durée de l'exercice 1, estimer `facteur_vitesse_maximal` : le facteur par lequel multiplier toutes les vitesses de synthèse pour que **toutes** les entrées tiennent sous la borne sans prolongation. Le comparer aux vitesses retenues pour l'intelligibilité dans les exemples guidés 2 et 2b : laquelle des deux contraintes est la plus sévère ?

**Indices** :
- `# Indice` : à la vitesse `v`, une réplique qui dure `d` à 1.0 dure environ `d / v` ; sa vitesse de lecture est donc multipliée par `v`
- `# Indice` : vérifiez qu'aucune entrée de `horaires_prolonges` ne chevauche la suivante : un chevauchement afficherait deux sous-titres à la fois
- `# Indice` : une entrée prolongée juste assez pour tenir la borne peut ressortir à 17,000000000000004 caractères par seconde en virgule flottante ; comparez avec une petite tolérance (`cps > BORNE_LECTURE + 1e-9`), sinon le décompte de `encore_trop_rapides` dépend de l'arrondi

In [31]:
# Exercice 3bis : vitesse de lecture -- le sous-titre laisse-t-il le temps de lire ?
# TODO etudiant : mesurer les caracteres par seconde des entrees de horaires_3 (exemple guide 6)
# Etape 1 : len("X : replique") / (fin - debut) -> cps_initial (liste), trop_rapides (nombre > borne)
# Etape 2 : prolonger dans le silence suivant (0,1 s de marge ; 0,5 s apres l'audio pour la derniere)
#           -> horaires_prolonges, encore_trop_rapides
# Etape 3 : loi de duree de l'exercice 1 -> facteur_vitesse_maximal pour tenir la borne partout
# Indice : a la vitesse v, une replique de duree d a 1.0 dure environ d / v
BORNE_LECTURE = 17  # caracteres par seconde
cps_initial = None  # TODO etudiant
trop_rapides = None  # TODO etudiant
horaires_prolonges = None  # TODO etudiant
encore_trop_rapides = None  # TODO etudiant
facteur_vitesse_maximal = None  # TODO etudiant
print("Exercice 3bis a completer : vitesse de lecture des sous-titres")


Exercice 3bis a completer : vitesse de lecture des sous-titres


## Conclusion

**Exercice 1.** Après correction du rééchantillonnage à 16 kHz, et sur les deux textes de l'exercice, 1,25 est la vitesse la plus rapide **parmi celles testées** dont le WER reste sous 10 % ; le premier dépassement arrive à 1,5 pour les deux textes. Le texte difficile ne se dégrade pas plus tôt, mais ses erreurs touchent l'information (noms propres, montant). Le protocole élargi précise la zone entre 1,25 et 1,5 sur un corpus de 123 mots, avec deux décodages : sa vitesse retenue est affichée par sa cellule et dans la synthèse automatique ci-dessous.

**Exercice 2.** Les accents suppriment les deux bascules vers l'anglais et corrigent aussi des prononciations qui ne signalaient aucune bascule. Le plan factoriel sépare les effets de la voix, de la langue de phonémisation et de l'orthographe, que la comparaison initiale confondait.

**Exercice 3.** Les deux fichiers SRT se terminent à moins de 0,5 ms de la fin de leur WAV ; l'alignement est garanti par construction, réplique par réplique.

**Portée.** Ces constats reposent sur des mesures automatiques (WER de Whisper, phonèmes, durées) et sur un avis d'écoute individuel. Ils valent pour les textes, les voix et les moteurs utilisés ici, en anglais américain, anglais britannique et français. La comparaison avec OpenAI n'est interprétable que si sa cellule l'a réellement mesurée. Le local évite la facturation par requête, mais conserve un coût matériel et énergétique.

### Prochaines étapes

- Explorer le TTS avance avec Chatterbox (02-1)
- Decouvrir le voice cloning avec XTTS (02-2)
- Construire un pipeline vocal complet (03-2)


## Section 6 : Pour aller plus loin - la limite du TTS ultra-leger (Inflect-Nano)

Kokoro (82 millions de paramètres) est déjà considere comme un modèle TTS "leger". Mais jusqu'ou peut-on reduire la taille d'un modèle de synthese vocale tout en produisant un audio comprehensible ? **Inflect-Nano-v1** (owensong, licence Apache-2.0) pousse l'exercice a l'extreme : **4,63 millions de paramètres au total**, soit ~18x plus petit que Kokoro et ~100x plus petit qu'un XTTS v2 (~466M).

### Architecture

Inflect-Nano decompose le TTS en deux reseaux non-autoregressifs (style FastSpeech), chacun deliberately minuscule :

| Composant | Rôle | Paramètres |
|-----------|------|------------|
| **Acoustique** (`MicroFastSpeech`) | Texte -> spectrogramme mel (80 bins) | 3,47 M |
| **Vocodeur** (`HifiGan` Snake) | Spectrogramme mel -> forme d'onde 24 kHz | 1,17 M |
| **Total** | | **4,63 M** |

Le pipeline complet : `texte -> front-end G2P anglais -> acoustic -> mel -> vocodeur -> 24 kHz`. L'inference est **non-autoregressive** (FastSpeech predit toutes les frames en parallele), ce qui rend le modèle extremement rapide : la cellule suivante mesure le rapport temps réel de la synthèse.

### Pourquoi s'y interesser ?

Ce modèle n'est **pas SOTA** - l'auteur le dit explicitement (qualite voice "robotique", voix masculine unique, anglais uniquement). Son intérêt est **pedagogique** : il demontre la frontiere inferieure de ce qu'un TTS neuronal peut faire. C'est le pendant du "perceptron minimal" pour la synthese vocale. Les cas d'usage listes par l'auteur : recherche, prototypage embarque, enseignement de l'architecture TTS. Les usages explicitement **deconseilles** : production, accessibilite, clonage vocal, multilingue, livre audio.

### Comparaison avec Kokoro

| Critere | Kokoro (82M) | Inflect-Nano (4,63M) |
|---------|--------------|----------------------|
| **Taille** | ~340 Mo (ONNX fp32 + voix) | ~49 Mo au total |
| **Paramètres** | 82 000 000 | 4 632 202 |
| **Ratio** | reference (1x) | ~18x plus petit |
| **Langues** | FR, EN, ES, IT, ZH, JP... | Anglais uniquement |
| **Voix** | 54 voix (ligne `Voix disponibles` de la cellule de chargement) | 1 voix masculine |
| **Architecture** | Style StyleTTS2 + plBERT | FastSpeech + HiFi-GAN Snake |
| **Qualite** | MOS ~4.0 (naturelle) | Robotique, demonstratif |
| **Matériel** | CPU suffisant, GPU facultatif | CPU suffisant |
| **Cas d'usage** | Production leger, multi-langue | Recherche, enseignement, embarque |

> **A retenir** : la taille n'est pas l'unique dimension. Kokoro investit ses 82M dans le multilinguisme, le multi-voix et le naturel. Inflect-Nano sacrifie tout sauf la fonctionnalite de base (produire de la parole comprehensible) pour atteindre 4,63M. Les deux sont "petits" mais pour des objectifs opposes.

In [32]:
# Section 6 facultative : exécutée seulement si run_inflect_nano = True
inflect_loaded = False
if not run_inflect_nano:
    print("Section facultative désactivée (run_inflect_nano = False) : non exécutée.")
else:
    # Demonstration Inflect-Nano-v1 : TTS ultra-leger (4,63M parametres)
    print("INFLECT-NANO-V1 - TTS ULTRA-LEGER")
    print("=" * 45)

    inflect_loaded = False
    inflect_samples = None
    inflect_sample_rate = 24000
    inflect_total_params = 4_632_202  # documente par l'auteur (3,47M acoustic + 1,17M vocoder)

    try:
        try:  # ressources nltk utilisées par g2p-en (noms selon la version de nltk)
            import nltk
            for ressource in ('averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger', 'cmudict'):
                nltk.download(ressource, quiet=True)
        except ImportError:
            pass
        from huggingface_hub import snapshot_download
        import torch, math

        # Telechargement du snapshot HF (deja en cache si deja telecharge)
        print("Recuperation du modele Inflect-Nano-v1...")
        snap_dir = snapshot_download(
            repo_id="owensong/Inflect-Nano-v1",
            allow_patterns=["*.py", "weights/*", "third_party/*"],
        )
        snap = Path(snap_dir)
        sys.path.insert(0, str(snap))
        sys.path.insert(0, str(snap / "third_party" / "tiny_tts_frontend"))

        # Imports (apres path setup)
        from tiny_tts.nn import commons
        from tiny_tts.text import phonemes_to_ids
        from tiny_tts.text.english import grapheme_to_phoneme, normalize_text
        from tiny_tts.utils import ADD_BLANK
        from inflect_nano.text_cleaning import clean_tinytts_text
        from inflect_nano.vocoder import HifiGanGenerator, make_config
        from inflect_nano.acoustic import MicroFastSpeech, MicroFastSpeechConfig

        device = torch.device("cuda" if gpu_available else "cpu")
        print(f"Backend : {device}")

        # Chargement acoustic
        ac = torch.load(snap / "weights" / "inflect_nano_v1_acoustic.pt",
                        map_location=device, weights_only=False)
        acfg = MicroFastSpeechConfig(**ac["config"])
        amodel = MicroFastSpeech(acfg).to(device); amodel.load_state_dict(ac["model"]); amodel.eval()
        acoustic_params = int(ac.get("params") or sum(p.numel() for p in amodel.parameters()))

        # Chargement vocoder
        vc = torch.load(snap / "weights" / "inflect_nano_v1_vocoder.pt",
                        map_location=device, weights_only=False)
        vcfg = make_config((vc.get("config") or {}).get("variant", "snake_v2mid"))
        vmodel = HifiGanGenerator(vcfg).to(device); vmodel.load_state_dict(vc["generator"])
        vmodel.remove_weight_norm(); vmodel.eval()
        vocoder_params = int(vc.get("generator_params") or sum(p.numel() for p in vmodel.parameters()))
        speakers = ac.get("speakers") or {"mark": 0}

        measured_total = acoustic_params + vocoder_params
        print(f"Params : acoustic={acoustic_params:,}  vocoder={vocoder_params:,}  total={measured_total:,}")
        assert measured_total == inflect_total_params, f"Ecart de compte: {measured_total} vs {inflect_total_params}"
        inflect_loaded = True

        # Generation (texte anglais - Inflect-Nano ne supporte que l'anglais)
        inflect_text = (
            "Kokoro weighs eighty two million parameters. "
            "Inflect Nano uses only four point six million."
        )
        print(f"Texte (anglais) : {inflect_text}")

        def _text_to_tokens(text):
            cleaned = clean_tinytts_text(text); normalized = normalize_text(cleaned)
            phones, tones, _ = grapheme_to_phoneme(normalized)
            pid, tid, lid = phonemes_to_ids(phones, tones, "EN")
            if ADD_BLANK:
                pid = commons.insert_blanks(pid, 0); tid = commons.insert_blanks(tid, 0); lid = commons.insert_blanks(lid, 0)
            return (torch.LongTensor(pid), torch.LongTensor(tid), torch.LongTensor(lid))

        phone, tone, lang = _text_to_tokens(inflect_text)
        phone = phone.unsqueeze(0).to(device); tone = tone.unsqueeze(0).to(device); lang = lang.unsqueeze(0).to(device)
        speaker = torch.LongTensor([int(speakers.get("mark", 0))]).to(device)

        t0 = time.time()
        with torch.inference_mode():
            mel = amodel.infer(phone, tone, lang, speaker, length_scale=1.0, pitch_scale=1.0, energy_scale=1.0)
            inflect_samples = vmodel(mel).squeeze().detach().cpu().numpy()
        gen_time = time.time() - t0

        # Normalisation (replication de inference.py officiel)
        def _rms_db(a): return 20.0 * math.log10(float(np.sqrt(np.mean(a**2, dtype=np.float64))) + 1e-9)
        inflect_samples = np.asarray(inflect_samples, dtype=np.float32).reshape(-1)
        inflect_samples = inflect_samples - float(inflect_samples.mean())
        inflect_samples *= 10 ** ((-20.0 - _rms_db(inflect_samples)) / 20.0)
        pk = float(np.max(np.abs(inflect_samples)) + 1e-9); lim = 10 ** (-1.0 / 20.0)
        if pk > lim: inflect_samples *= lim / pk
        inflect_samples = np.clip(inflect_samples, -1.0, 1.0)

        duration = len(inflect_samples) / inflect_sample_rate
        print(f"\nGeneration reussie")
        print(f"  Duree audio : {duration:.2f}s")
        print(f"  Temps CPU : {gen_time:.2f}s  (RTF = {gen_time/duration:.3f}, < 1 = plus rapide que le temps reel)")
        print(f"\nEcoute (notez le caractere robotique, voix masculine unique) :")
        display_audio_array(inflect_samples, inflect_sample_rate)

        if save_results:
            out = OUTPUT_DIR / "inflect_nano_demo.wav"
            sf.write(str(out), inflect_samples, inflect_sample_rate, subtype="PCM_16")
            print(f"Fichier sauvegarde : {out.name}")

    except ImportError as e:
        print(f"Dependance manquante ({e}). Requiert : torch, soundfile, g2p_en, huggingface_hub.")
        print("Inflect-Nano n'est pas installe - demonstration sautee (graceful).")
    except Exception as e:
        print(f"Inflect-Nano indisponible sur cette machine : {str(e)[:120]}")
        print("Demonstration non executee : l'indiquer comme « non exécutée » dans l'interpretation.")


INFLECT-NANO-V1 - TTS ULTRA-LEGER
Recuperation du modele Inflect-Nano-v1...


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/owensong/Inflect-Nano-v1/revision/main "HTTP/1.1 200 OK"


Fetching 32 files:   0%|          | 0/32 [00:00<?, ?it/s]

Fetching 32 files: 100%|██████████| 32/32 [00:00<00:00, 1705.28it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"


... (4 outputs left out)

Backend : cpu
Params : acoustic=3,465,125  vocoder=1,167,077  total=4,632,202
Texte (anglais) : Kokoro weighs eighty two million parameters. Inflect Nano uses only four point six million.



Generation reussie
  Duree audio : 4.49s
  Temps CPU : 0.69s  (RTF = 0.154, < 1 = plus rapide que le temps reel)

Ecoute (notez le caractere robotique, voix masculine unique) :


<audio embedded (audio/wav, 215596 bytes)>

Fichier sauvegarde : inflect_nano_demo.wav


### Extension facultative : interprétation

Cette démonstration ne constitue une mesure que si la cellule précédente aboutit (`inflect_loaded = True`). Sinon, elle est « non exécutée », et la synthèse automatique ci-dessous l'indique. Si elle aboutit : comparer la durée et le temps de calcul à ceux de Kokoro, et écouter le résultat. La taille seule ne permet pas de déduire un seuil universel de qualité.


## Synthèse automatique de l'exécution

La cellule suivante résume les résultats **calculés pendant cette exécution** : les valeurs et les statuts (« mesuré », « non mesuré », « non exécutée ») viennent des variables du notebook, pas d'un texte rédigé à l'avance.


In [33]:
# Synthèse automatique : résumé calculé à partir des résultats de CETTE exécution
print("SYNTHÈSE AUTOMATIQUE DE L'EXÉCUTION")
print("=" * 45)

print("\nExercice 1 (deux textes, sept vitesses) :")
for nom in textes:
    lignes = [r for r in mesures if r['texte'] == nom]
    retenue = vitesse_retenue({r['vitesse']: r['wer'] for r in lignes}, SEUIL_WER)
    print(f"  {nom:10} : vitesse la plus rapide testée sous {SEUIL_WER:.0%} = {retenue} | WER : "
          + ", ".join(f"{r['vitesse']}→{r['wer']:.0%}" for r in lignes))
print("  Effet du rééchantillonnage : voir effet_reechantillonnage.csv")
if synthese_corpus:
    print(f"  Protocole élargi : vitesse la plus rapide parmi celles testées sur ce corpus = {vitesse_corpus}")
else:
    print("  Protocole élargi : non mesuré")

print("\nExercice 2 :")
print("  Bascules vers l'anglais (fr-fr) sans / avec accents :",
      [a['fr'].count('(en)') for a in analyses], "/", [a['fr_corrige'].count('(en)') for a in analyses])
print(f"  Épisode : {len(manifeste['articles'])} articles, {manifeste['duration_seconds']:.3f} s")
if effets_factoriel:
    print("  Plan factoriel, écart moyen de WER dû à chaque facteur :",
          ", ".join(f"{f} {e:+.1%}" for f, e in effets_factoriel.items()))
else:
    print("  Plan factoriel : non mesuré")

print("\nExercice 3 :")
print(f"  {len(horaires_initial)} puis {len(horaires_3)} répliques ; fins SRT {horaires_initial[-1][3]:.3f} s "
      f"et {horaires_3[-1][3]:.3f} s ; alignement vérifié par assertions")

print("\nSection 4 (comparaison cloud) :")
for l in comparaison:
    print(f"  {l['service']:<32} {l['statut']}" + (f" ({l['remarque']})" if l['statut'] != 'mesuré' else ""))

print("\nSection 6 (Inflect-Nano) :", "exécutée" if globals().get('inflect_loaded') else "non exécutée")


SYNTHÈSE AUTOMATIQUE DE L'EXÉCUTION

Exercice 1 (deux textes, sept vitesses) :
  simple     : vitesse la plus rapide testée sous 10% = 1.25 | WER : 0.5→0%, 0.75→0%, 1.0→0%, 1.25→0%, 1.5→18%, 1.75→35%, 2.0→41%
  difficile  : vitesse la plus rapide testée sous 10% = 1.25 | WER : 0.5→5%, 0.75→0%, 1.0→0%, 1.25→0%, 1.5→25%, 1.75→20%, 2.0→20%
  Effet du rééchantillonnage : voir effet_reechantillonnage.csv
  Protocole élargi : vitesse la plus rapide parmi celles testées sur ce corpus = 1.5

Exercice 2 :
  Bascules vers l'anglais (fr-fr) sans / avec accents : [0, 1, 1] / [0, 0, 0]
  Épisode : 3 articles, 16.656 s
  Plan factoriel, écart moyen de WER dû à chaque facteur : voix -5.8%, langue -2.5%, orthographe -27.5%

Exercice 3 :
  6 puis 8 répliques ; fins SRT 15.897 s et 21.228 s ; alignement vérifié par assertions

Section 4 (comparaison cloud) :
  Kokoro ff_siwis (local, CPU)     mesuré
  OpenAI tts-1 (voix alloy)        mesuré
  OpenAI tts-1-hd (voix alloy)     mesuré

Section 6 (Inflect-N

## Évaluation auditive personnelle

**Statut : réalisée.** Il s'agit d'un avis individuel, pas d'un MOS calculé sur un panel d'auditeurs.

- **Évaluateur** : ⟨prénom et nom⟩, seul évaluateur.
- **Conditions d'écoute** : ⟨casque, écouteurs ou haut-parleurs⟩, lecteur audio intégré au notebook, ⟨environnement calme ou non⟩, le ⟨date⟩.
- **Fichiers écoutés** : indiqués pour chaque ligne du tableau. Ils ont été produits par l'exécution V3. La V4 les régénère à l'identique (synthèse déterministe), sauf la première génération de la section 2, dont le texte est désormais accentué.
- **Non notés** : les nouveaux fichiers de la V4 (protocole élargi, plan factoriel, comparaison cloud). Une ligne OpenAI ne peut être ajoutée que si la comparaison a été réellement mesurée.

Chaque critère est noté de 1 à 5 (1 = très mauvais ; 2 = insuffisant ; 3 = moyen ; 4 = bon ; 5 = excellent).

| Comparaison | Fichiers écoutés | Naturalité /5 | Intelligibilité /5 | Prosodie /5 | Observation personnelle |
|---|---|---|---|---|---|
| Vitesse et confort | `speed_0.5` à `speed_2.0` (`_af_heart.wav`) | 4 | 5 | 4 | Les vitesses 1,0 et 1,25 offrent le meilleur compromis entre fluidité, naturel et intelligibilité. À 0,5, la voix est trop lente et manque de naturel. À partir de 1,5, le débit devient plus rapide et demande davantage d'attention. À 2,0, certaines articulations sont moins distinctes. Je privilégie la vitesse 1,25, qui permet une lecture rapide tout en conservant une bonne compréhension. |
| Noms propres et nombres | `difficile_speed_0.5` à `difficile_speed_2.0` (`_af_heart.wav`) | 4 | 4 | 4 | À vitesse normale (1,0), les noms propres, les nombres et le sigle GPU sont globalement compréhensibles. Les noms Nguyen et Siobhan O'Connor nécessitent une attention particulière en raison de leur prononciation. Les nombres 137 et 2 450 restent perceptibles, mais leur compréhension devient plus exigeante lorsque le débit augmente. À 1,5, les mots s'enchaînent rapidement et certaines articulations sont moins distinctes. À 2,0, l'écoute devient moins confortable, notamment pour identifier les noms et les valeurs numériques. La vitesse 1,0 reste préférable pour ce type de texte contenant des informations précises. |
| Prononciation française | `article_1` à `article_3`, en `_af_heart.wav` et `_ff_siwis.wav` | 4 | 5 | 4 | La comparaison des trois articles montre une nette différence entre les voix af_heart et ff_siwis. La voix af_heart présente un accent anglophone prononcé sur plusieurs mots français. La voix ff_siwis offre une prononciation française plus naturelle, avec des accents mieux respectés et une articulation plus fluide. L'épellation de « I.A. » est également plus intelligible dans la version française. Je privilégie donc ff_siwis pour les textes français. Toutefois, la voix, la langue de phonémisation et l'orthographe changent simultanément, ce qui ne permet pas d'attribuer ces améliorations à un seul facteur. |
| Timbre et naturel | `voice_af_heart.wav`, `voice_af_bella.wav`, `voice_am_adam.wav`, `voice_ff_siwis.wav` | 5 | 5 | 4 | L'écoute des quatre voix met en évidence des différences de timbre, de rythme et d'intonation. La voix af_heart possède un timbre féminin clair, doux et relativement naturel, avec une articulation précise. La voix ff_siwis présente une prononciation française fluide et agréable, adaptée aux textes francophones. La voix am_adam se distingue par un timbre masculin plus grave et posé, avec une diction claire. La voix af_bella possède un timbre féminin plus lumineux et expressif, avec une intonation dynamique. Les quatre voix restent facilement compréhensibles. Je privilégie af_heart pour la synthèse anglaise en raison de son équilibre entre naturel, clarté et fluidité, et ff_siwis pour la synthèse française. Cette comparaison reste subjective puisque les textes et les langues ne sont pas identiques. |
| Dialogue et sous-titres | `dialogue_3_voix.wav` et `dialogue_3_voix.srt` | 4 | 5 | 4 | Le dialogue présente trois voix distinctes, ce qui permet d'identifier facilement les différents locuteurs. Les changements de timbre entre les voix féminines et masculine rendent les échanges clairs et agréables à suivre. Les pauses entre les répliques sont régulières et facilitent la compréhension, même si elles donnent parfois un aspect légèrement artificiel à la conversation. L'intonation des questions et des exclamations est globalement satisfaisante, particulièrement dans les dernières répliques. Le fichier SRT contient huit sous-titres correspondant aux huit interventions, avec des horaires chronologiques et sans chevauchement. Les sous-titres suivent correctement la structure du dialogue, mais leur synchronisation exacte à l'écoute reste à confirmer dans un lecteur compatible. L'ensemble constitue une démonstration convaincante d'un dialogue synthétique à trois voix. |

Pour attribuer une différence de prononciation française à un seul facteur, écouter deux fichiers de `plan_factoriel/` qui ne diffèrent que par ce facteur (voir l'analyse de l'exercice 2).


In [34]:
# Export du rendu : notebook désigné explicitement + résultats ; fonctionne dans Colab et en local
# Avant d'exécuter : enregistrer le notebook (Ctrl+S), puis passer run_export à True.
NOTEBOOK_A_EXPORTER = "01-5-Kokoro_TTS_Local_V4.ipynb"  # nom exact, ou chemin complet du fichier
DOSSIER_EXPORT = Path.cwd() / "TP_Kokoro_TTS_Complet"
FICHIERS_ATTENDUS = ["intelligibilite.csv", "effet_reechantillonnage.csv", "phonemes.json", "episode.json",
                     "podcast_complet_fr.wav", "sigle_ia.csv", "comparaison_cloud.csv",
                     "dialogue.srt", "dialogue.wav", "dialogue_3_voix.srt", "dialogue_3_voix.wav"]
if run_protocole_elargi:
    FICHIERS_ATTENDUS += ["protocole_elargi.csv", "protocole_elargi_synthese.csv"]
if run_plan_factoriel:
    FICHIERS_ATTENDUS += ["plan_factoriel.csv"]

if not run_export:
    print("Export désactivé (run_export = False). Pour exporter : enregistrer le notebook, "
          "passer run_export à True dans la cellule des paramètres, l'exécuter, puis exécuter cette cellule.")
else:
    import shutil, platform, json
    from importlib.metadata import version, PackageNotFoundError
    try:
        from google.colab import drive, files
        en_colab = True
    except ImportError:
        en_colab = False

    # 1. Localiser exactement le notebook à rendre
    chemin_direct = Path(NOTEBOOK_A_EXPORTER)
    if chemin_direct.is_absolute() and chemin_direct.exists():
        candidats = [chemin_direct]
    else:
        candidats = [p for p in [Path.cwd() / NOTEBOOK_A_EXPORTER] if p.exists()]
        if en_colab and not candidats:
            drive.mount("/content/drive")
            candidats = sorted(Path("/content/drive/MyDrive").rglob(NOTEBOOK_A_EXPORTER))
    if not candidats:
        raise FileNotFoundError(f"« {NOTEBOOK_A_EXPORTER} » introuvable (dossier courant"
                                + (" et Google Drive" if en_colab else "") + "). Enregistrer le notebook "
                                "sous ce nom, ou indiquer son chemin complet dans NOTEBOOK_A_EXPORTER.")
    if len(candidats) > 1:
        raise RuntimeError("Plusieurs fichiers portent ce nom :\n" + "\n".join(map(str, candidats))
                           + "\nIndiquer le chemin complet du bon fichier dans NOTEBOOK_A_EXPORTER.")
    notebook = candidats[0]

    # 2. Copier le notebook et les résultats
    if DOSSIER_EXPORT.exists():
        shutil.rmtree(DOSSIER_EXPORT)
    (DOSSIER_EXPORT / "outputs" / "audio").mkdir(parents=True)
    shutil.copy2(notebook, DOSSIER_EXPORT / notebook.name)
    shutil.copytree(OUTPUT_DIR, DOSSIER_EXPORT / "outputs" / "audio" / "kokoro")
    print("Notebook exporté :", notebook)

    # 3. Vérifier les fichiers attendus
    manquants = [f for f in FICHIERS_ATTENDUS if not (OUTPUT_DIR / f).exists()]
    print("Fichiers attendus :", len(FICHIERS_ATTENDUS) - len(manquants), "présents sur", len(FICHIERS_ATTENDUS))
    if manquants:
        print("MANQUANTS :", ", ".join(manquants), "-> exécuter les cellules correspondantes avant d'exporter.")

    # 4. Environnement : versions installées et paramètres de l'exécution
    installees = {}
    for paquet in [*VERSIONS_VALIDEES, "g2p-en", "torch"]:
        try:
            installees[paquet] = version(paquet)
        except PackageNotFoundError:
            pass
    (DOSSIER_EXPORT / "requirements.txt").write_text(
        "\n".join(f"{p}=={v}" for p, v in installees.items() if p in VERSIONS_VALIDEES) + "\n"
        "# Facultatif (section 6) : g2p-en==2.1.0 et torch\n", encoding="utf-8")
    (DOSSIER_EXPORT / "versions_validation.json").write_text(json.dumps({
        "notebook": notebook.name, "date": datetime.now().isoformat(timespec="seconds"),
        "python": platform.python_version(), "plateforme": platform.platform(), "colab": en_colab,
        "versions_installees": installees, "versions_validees": VERSIONS_VALIDEES,
        "parametres": {k: globals()[k] for k in ["run_protocole_elargi", "run_plan_factoriel",
                       "run_cloud_comparison", "cloud_models", "cloud_repetitions", "run_inflect_nano"]},
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    # 5. README et .gitignore
    (DOSSIER_EXPORT / "README.md").write_text(f"""# TP Kokoro TTS Local (V4)

Synthèse vocale locale avec Kokoro-82M (ONNX), évaluée avec faster-whisper.

## Contenu
- `{notebook.name}` : notebook exécuté
- `outputs/audio/kokoro/` : audios WAV, CSV et JSON produits par le notebook
- `requirements.txt` et `versions_validation.json` : environnement de l'exécution

## Reproduire
1. Python 3.13, puis `pip install -r requirements.txt` (ou exécuter la cellule d'installation du notebook).
2. Ouvrir le notebook dans Jupyter ou Google Colab et tout exécuter dans l'ordre.
3. Aucune clé n'est nécessaire pour le parcours principal. La comparaison OpenAI demande
   `OPENAI_API_KEY` et `run_cloud_comparison = True` ; sans cela, elle affiche « non mesuré ».

## Périmètre et limites
- Langues testées : anglais américain, anglais britannique, français.
- Le WER mesure une transcription automatique, pas l'écoute humaine.
- Les temps de calcul dépendent du matériel.

## Licences
Kokoro-82M : Apache-2.0 ; kokoro-onnx : MIT ; faster-whisper : MIT.
""", encoding="utf-8")
    (DOSSIER_EXPORT / ".gitignore").write_text(".env\n__pycache__/\n.ipynb_checkpoints/\n.venv/\n.cache/\n*.onnx\nvoices-v1.0.bin\n",
                                               encoding="utf-8")

    # 6. Archive
    archive = shutil.make_archive(str(DOSSIER_EXPORT), "zip", root_dir=DOSSIER_EXPORT.parent,
                                  base_dir=DOSSIER_EXPORT.name)
    print("Archive créée :", archive)
    if en_colab:
        files.download(archive)
    else:
        print("Hors Colab : récupérer l'archive à l'emplacement ci-dessus.")


Export désactivé (run_export = False). Pour exporter : enregistrer le notebook, passer run_export à True dans la cellule des paramètres, l'exécuter, puis exécuter cette cellule.
